# 🎙️ Speech Corpus Pipeline: Student 1 (Standalone Colab Notebook, v14 — corrected)

Covers **Steps 1-4** of the project brief: download, audio extraction, speech/music/mixed/silence labelling, and 5-15 s cutting at real pauses.

**What changed vs. v13 (all fixes from the review):**
- Step 1: `info.json` metadata (link, title, channel, duration, license), correct Creative-Commons check, optional cookies file, no 20-min duration cap.
- Step 3: region boundaries no longer overlap; AudioSet ids come from `model.config.id2label`; models are cached; no silent heuristic fallback; silence comes from energy; minimum region length; boundaries snapped to VAD pauses; spectrogram editor with movable borders.
- Step 4: pause/margin logic reconciled (short pauses are flagged), stream edges snapped to pauses, short streams merged, segment labels inherit the (human-edited) region label, music regions included, better Whisper word check (Arabic, tolerant to timestamp noise), measured greedy-vs-DP ablation.
- QC: only measured numbers, real check counts, manual 10-minute check tool, per-video summary, safe splitting.
- Speed: vectorised energy computation.

In [ ]:
# 📦 Cell 1: Environment Setup
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("⚡ Running in Google Colab. Installing dependencies...")
    !pip install -q yt-dlp faster-whisper transformers soundfile librosa scikit-learn ipywidgets
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except Exception:
        pass
    print("✅ Colab environment configured.")
else:
    print("💻 Local Jupyter environment. Dependencies assumed installed.")

In [ ]:
# ⚙️ Cell 2: Global Imports & Constants
import os, io, re, sys, json, time, math, base64, bisect, shutil, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt

try:
    import torch
except ImportError:          # torch is only needed for AST / Silero
    torch = None
from IPython.display import display, Audio, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- audio / labels -------------------------------------------------------
DEFAULT_SR = 16000
CONTENT_LABELS = ["speech", "music", "mixed", "silence"]

# ---- Step 3 (AST + regions) ----------------------------------------------
AST_WIN_S = 10.0             # AST was trained on ~10 s clips
AST_HOP_S = 5.0
MIXED_SP_THR = 0.40          # mixed = speech AND music both present (tune on your hand-labelled 10 min)
MIXED_MU_THR = 0.30
SILENCE_DB_ABOVE_FLOOR = 4.0 # window quieter than floor+4 dB => silence (energy decides silence, not AST)
MIN_REGION_S = 2.0           # regions shorter than this are merged into a neighbour
SNAP_WINDOW_S = 2.5          # region borders snap to a VAD pause within +-2.5 s
ALLOW_HEURISTIC_FALLBACK = False   # if AST cannot load: stop instead of silently using RMS/ZCR

# ---- Step 4 (Cut Engine v2) ----------------------------------------------
MIN_LEN = 5.0
MAX_LEN = 15.0
FB_LEN = 18.0
TARGET = 10.0
LAM = 1.0                    # weight of the (normalised) length prior ((L-TARGET)/TARGET)^2
CUT_W = 0.5                  # weight of cut quality: cost (1-score) per cut. v13 REWARDED cuts with -score, which
                             # pushed the DP towards many short segments; a cost makes it prefer good cuts, not more cuts
FB_PEN = 10.0
WEAK_PEN = 2.0
SHORT_PEN = 0.5              # extra cost for pauses < 200 ms (cannot give 100 ms margin on both sides)
MARGIN_MAX = 0.20
MARGIN_MIN = 0.10
MARGIN_TOL_MS = 3
MIN_PAUSE_CAND = 0.12        # shortest pause considered as a cut candidate
MIN_PAUSE_STRONG = 0.20      # 2 x MARGIN_MIN: shortest pause that can satisfy the 100-200 ms rule
STREAM_MERGE_GAP = 1.2
SHORT_STREAM_GAP = 3.0
WORD_TS_TOL = 0.15           # Whisper word timestamps are only good to ~100-200 ms
HIGH_TIER = 0.75
MED_TIER = 0.50

if torch is not None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_name = torch.cuda.get_device_name(0) if device == "cuda" else "Host CPU"
else:
    device, dev_name = "cpu", "Host CPU (torch missing)"
print(f"🚀 Device: {device.upper()} ({dev_name})")

In [ ]:
# 🎛️ Cell 3: Audio Ingestion (Steps 1-2)

def check_ffmpeg() -> bool:
    return shutil.which("ffmpeg") is not None


def read_wav(path, target_sr: int = DEFAULT_SR):
    """Read a WAV as mono float32; polyphase-resample if needed (fast, no FFT ringing)."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Audio file not found: {path}")
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)
    mono = data.mean(axis=1) if data.shape[1] > 1 else data[:, 0]
    if sr != target_sr:
        g = math.gcd(int(sr), int(target_sr))
        mono = signal.resample_poly(mono, int(target_sr) // g, int(sr) // g).astype(np.float32)
        sr = target_sr
    return np.ascontiguousarray(mono), sr


def write_wav(path, audio: np.ndarray, sample_rate: int = DEFAULT_SR):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if audio.dtype != np.int16:
        pcm16 = np.round(np.clip(audio, -1.0, 1.0) * 32767.0).astype(np.int16)
    else:
        pcm16 = audio
    sf.write(str(path), pcm16, sample_rate, subtype="PCM_16")


def extract_audio_hq(src, dst_wav, sr: int = DEFAULT_SR) -> Path:
    """ffmpeg -> 16 kHz / mono / 16-bit PCM WAV (ffmpeg does the resampling)."""
    src, dst_wav = Path(src), Path(dst_wav)
    dst_wav.parent.mkdir(parents=True, exist_ok=True)
    if check_ffmpeg():
        cmd = ["ffmpeg", "-y", "-i", str(src), "-vn", "-acodec", "pcm_s16le",
               "-ar", str(sr), "-ac", "1", str(dst_wav)]
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"FFmpeg extraction failed: {res.stderr[-800:]}")
    else:
        mono, _ = read_wav(src, target_sr=sr)
        write_wav(dst_wav, mono, sample_rate=sr)
    return dst_wav


def extract_video_id(url_or_path: str) -> str:
    m = re.search(r"(?:v=|\/)([0-9A-Za-z_-]{11})(?:\?|&|$|\/)", url_or_path)
    if m:
        return m.group(1)
    name = re.sub(r"[^A-Za-z0-9_-]", "_", Path(url_or_path).stem)
    return name[:32] if name else "sample_video"


def is_creative_commons(info: dict) -> bool:
    """True only if yt-dlp's full metadata says 'Creative Commons' (no loose 'cc' substring match)."""
    return "creative commons" in str((info or {}).get("license") or "").lower()


def _save_info(out_dir: Path, info: dict) -> dict:
    (out_dir / "info.json").write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    return info


def download_video(url: str, out_dir, max_duration: Optional[int] = None, cookies_file=None):
    """Download best audio, convert to 16 kHz WAV, save info.json. Returns (wav_path, info_dict).
    max_duration=None means no cap (the project needs videos >= 1 hour)."""
    import yt_dlp
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_path, info_path = out_dir / "audio.wav", out_dir / "info.json"

    if wav_path.exists() and info_path.exists():
        print(f"⚡ Cached audio + info found in {out_dir}")
        return wav_path, json.loads(info_path.read_text(encoding="utf-8"))

    base = {"quiet": True, "no_warnings": True, "noplaylist": True}
    if cookies_file:
        base["cookiefile"] = str(cookies_file)

    with yt_dlp.YoutubeDL(base) as ydl:
        meta = ydl.extract_info(url, download=False)
    dur = meta.get("duration")
    if max_duration and dur and dur > max_duration:
        raise ValueError(f"Video is {dur}s, longer than max_duration={max_duration}s")

    info = {
        "video_id": extract_video_id(url), "source": "youtube", "url": url,
        "webpage_url": meta.get("webpage_url"), "title": meta.get("title"),
        "channel": meta.get("channel") or meta.get("uploader"),
        "duration_s": dur, "upload_date": meta.get("upload_date"),
        "license": meta.get("license"), "is_creative_commons": is_creative_commons(meta),
        "downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    if not info["is_creative_commons"]:
        print(f"⚠️ License is '{info['license']}', not Creative Commons. Document this in the report.")

    if not wav_path.exists():
        opts = dict(base, format="bestaudio/best", outtmpl=str(out_dir / "temp_video.%(ext)s"))
        with yt_dlp.YoutubeDL(opts) as ydl:
            ydl.download([url])
        temp = list(out_dir.glob("temp_video.*"))
        if not temp:
            raise RuntimeError("yt-dlp produced no file (on Colab: YouTube may block the IP; pass cookies_file or upload the audio).")
        extract_audio_hq(temp[0], wav_path)
        for f in temp:
            f.unlink(missing_ok=True)
    _save_info(out_dir, info)
    return wav_path, info


def ingest_local_file(src, out_dir):
    """Use a local audio/video file instead of a URL. Returns (wav_path, info_dict)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    wav_path = extract_audio_hq(src, out_dir / "audio.wav")
    dur = sf.info(str(wav_path)).duration
    info = {"video_id": out_dir.name, "source": "local_file", "url": str(src), "title": Path(src).name,
            "channel": None, "duration_s": round(dur, 2), "license": "unknown",
            "is_creative_commons": False, "downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    return wav_path, _save_info(out_dir, info)


def build_mock_audio(seed: int = 42, sr: int = DEFAULT_SR):
    """Synthetic multi-class audio for an instant offline smoke test."""
    rng = np.random.default_rng(seed)
    timeline = [("silence", 3.0), ("speech", 14.0), ("music", 10.0), ("mixed", 14.0), ("silence", 2.0),
                ("speech", 12.0), ("music", 8.0), ("mixed", 12.0), ("silence", 3.0)]
    parts, rows, t = [], [], 0.0
    for lab, d in timeline:
        n = int(d * sr); tt = np.arange(n) / sr
        if lab == "silence":
            x = 0.0002 * rng.standard_normal(n)
        elif lab == "speech":
            x = (0.3 * np.sin(2 * np.pi * 180 * tt) + 0.15 * np.sin(2 * np.pi * 700 * tt)) * (np.sin(2 * np.pi * 0.7 * tt) ** 2 > 0.05)
        elif lab == "music":
            x = (0.2 * np.sin(2 * np.pi * 440 * tt) + 0.15 * np.sin(2 * np.pi * 554.37 * tt) + 0.15 * np.sin(2 * np.pi * 659.25 * tt)) * 0.4
        else:
            x = ((0.25 * np.sin(2 * np.pi * 180 * tt) + 0.1 * np.sin(2 * np.pi * 700 * tt)) * (np.sin(2 * np.pi * 0.7 * tt) ** 2 > 0.05) + 0.1 * np.sin(2 * np.pi * 440 * tt)) * 0.5
        parts.append(x.astype(np.float32))
        rows.append({"start": round(t, 2), "end": round(t + d, 2), "label": lab, "duration": round(d, 2)})
        t += d
    return np.concatenate(parts), pd.DataFrame(rows)

print("✅ Audio ingestion module ready.")

In [ ]:
# ⚡ Cell 4: Energy, Adaptive Silence Floor, VAD Pauses

def compute_energy_db(audio: np.ndarray, sr: int = DEFAULT_SR, frame_ms: float = 25.0, hop_ms: float = 10.0):
    """Vectorised short-time energy (dB). Returns (frame_start_times, energy_db). ~100x faster than a Python loop."""
    frame_len, hop = int(sr * frame_ms / 1000), int(sr * hop_ms / 1000)
    if len(audio) < frame_len:
        audio = np.pad(audio, (0, frame_len - len(audio)))
    c = np.concatenate(([0.0], np.cumsum(audio.astype(np.float64) ** 2)))
    idx = np.arange(0, len(audio) - frame_len + 1, hop)
    ms = np.maximum((c[idx + frame_len] - c[idx]) / frame_len, 0.0)
    return idx / sr, (10.0 * np.log10(ms + 1e-14)).astype(np.float32)


def adaptive_silence_floor(energy_db: np.ndarray):
    """Otsu threshold on the dB histogram (separates quiet/ambient frames from loud frames).
    Returns (floor_db = median of the quiet class, threshold_db)."""
    e = np.clip(energy_db, -100.0, 0.0)
    lo, hi = np.percentile(e, [1, 99])
    if hi - lo < 3.0:
        return float(lo), float(lo + 8.5)
    edges = np.arange(math.floor(lo), math.ceil(hi) + 1, 1.0)
    hist, edges = np.histogram(e, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2.0
    w = hist.astype(float)
    w0, m = np.cumsum(w), np.cumsum(w * centers)
    w1 = w.sum() - w0
    with np.errstate(divide="ignore", invalid="ignore"):
        between = w0 * w1 * (m / w0 - (m[-1] - m) / w1) ** 2
    thr = float(centers[int(np.argmax(np.nan_to_num(between)))])
    low = e[e <= thr]
    floor = float(np.median(low)) if low.size else float(lo)
    return floor, float(np.clip(thr, floor + 3.0, floor + 20.0))


def _true_runs(mask: np.ndarray):
    m = np.concatenate(([False], np.asarray(mask, dtype=bool), [False])).astype(np.int8)
    d = np.diff(m)
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))   # [start_idx, end_idx_exclusive)


class PauseInterval:
    def __init__(self, start: float, end: float, center: Optional[float] = None):
        self.start = round(float(start), 3)
        self.end = round(float(end), 3)
        self.center = round(float(center if center is not None else (start + end) / 2.0), 3)
        self.duration = round(self.end - self.start, 3)
        self.duration_ms = round(self.duration * 1000.0, 1)


_SILERO = {}
VAD_BACKEND_USED = {"name": None}


def _load_silero():
    if "m" not in _SILERO:
        import os
        cache_dir = os.path.expanduser("~/.cache/torch/hub/snakers4_silero-vad_master")
        if os.path.exists(cache_dir):
            try:
                model, utils = torch.hub.load(cache_dir, "silero_vad", source="local", trust_repo=True)
                _SILERO["m"] = (model, utils[0])
                return _SILERO["m"]
            except Exception:
                pass
        model, utils = torch.hub.load(repo_or_dir="snakers4/silero-vad", model="silero_vad", trust_repo=True)
        _SILERO["m"] = (model, utils[0])
    return _SILERO["m"]


def detect_pauses_energy(audio, sr=DEFAULT_SR, min_pause_ms=120):
    times, e = compute_energy_db(audio, sr)
    _, thr = adaptive_silence_floor(e)
    total = len(audio) / sr
    pauses = []
    for a, b in _true_runs(e < thr):
        t0 = float(times[a]); t1 = float(times[b]) if b < len(times) else total   # closes a pause open at EOF
        if t1 - t0 >= min_pause_ms / 1000.0:
            pauses.append(PauseInterval(t0, t1))
    return pauses


def detect_pauses_silero(audio, sr=DEFAULT_SR, min_pause_ms=120, speech_pad_ms=10, vad_backend="auto"):
    """Pauses = gaps between Silero speech segments. speech_pad_ms is explicit (library default of 30 ms would
    shrink every pause by 60 ms, which matters for the 100-200 ms margin rule). Logs which backend was used."""
    total = len(audio) / sr
    if vad_backend != "energy" and torch is not None:
        try:
            model, get_ts = _load_silero()
            ts = get_ts(torch.from_numpy(audio).float(), model, sampling_rate=sr, threshold=0.5,
                        min_silence_duration_ms=min_pause_ms, min_speech_duration_ms=100,
                        speech_pad_ms=speech_pad_ms)
            VAD_BACKEND_USED["name"] = "silero"
            if not ts:
                return [PauseInterval(0.0, total)]
            pauses, cur = [], 0.0
            for s in ts:
                s0, s1 = s["start"] / sr, s["end"] / sr
                if s0 - cur >= min_pause_ms / 1000.0:
                    pauses.append(PauseInterval(cur, s0))
                cur = s1
            if total - cur >= min_pause_ms / 1000.0:
                pauses.append(PauseInterval(cur, total))
            return pauses
        except Exception as e:
            print(f"⚠️ Silero VAD unavailable ({type(e).__name__}: {e}). Falling back to ENERGY pause detector.")
    VAD_BACKEND_USED["name"] = "energy"
    return detect_pauses_energy(audio, sr, min_pause_ms)


def detect_speech_activity_overlay(audio, sr=DEFAULT_SR, silence_threshold_s=0.25, min_pause_ms=120, vad_backend="auto"):
    """Combines neural Silero VAD with Short-Time RMS Energy Floor to capture genuine
    acoustic pauses even in mixed/music-accompanied speech stretches.
    """
    total = len(audio) / sr
    pauses_silero = detect_pauses_silero(audio, sr, min_pause_ms=min_pause_ms, vad_backend=vad_backend)
    pauses_energy = detect_pauses_energy(audio, sr, min_pause_ms=min_pause_ms)

    merged = list(pauses_silero)
    for ep in pauses_energy:
        if not any(max(ep.start, sp.start) < min(ep.end, sp.end) for sp in merged):
            merged.append(ep)

    pauses = sorted(merged, key=lambda p: p.start)
    total = len(audio) / sr
    timeline, cur = [], 0.0
    for p in pauses:
        if p.start > cur:
            timeline.append({"start": round(cur, 3), "end": p.start, "state": "SPEECH", "duration": round(p.start - cur, 3)})
        timeline.append({"start": p.start, "end": p.end, "state": "SILENCE", "duration": p.duration})
        cur = p.end
    if cur < total:
        timeline.append({"start": round(cur, 3), "end": round(total, 3), "state": "SPEECH", "duration": round(total - cur, 3)})
    rec = [{"pause_id": f"P{i:04d}", "start": p.start, "end": p.end, "center": p.center, "duration": p.duration,
            "duration_ms": p.duration_ms, "is_safe": p.duration >= silence_threshold_s} for i, p in enumerate(pauses, 1)]
    cols = ["pause_id", "start", "end", "center", "duration", "duration_ms", "is_safe"]
    return pd.DataFrame(timeline), pd.DataFrame(rec, columns=cols)


def compute_instantaneous_energy_vad_overlay(audio, sr=DEFAULT_SR, vad_timeline_df=None):
    """Returns (intervals_df, times, energy_db, thresh_db). thresh_db = adaptive silence threshold (floor + margin)."""
    times, energy_db = compute_energy_db(audio, sr)
    floor_db, thresh_db = adaptive_silence_floor(energy_db)
    if vad_timeline_df is not None and not vad_timeline_df.empty:
        return vad_timeline_df, times, energy_db, thresh_db
    total = len(audio) / sr
    rows = []
    sp = energy_db > thresh_db
    bounds = np.concatenate(([0], np.where(np.diff(sp.astype(np.int8)) != 0)[0] + 1, [len(sp)]))
    for a, b in zip(bounds[:-1], bounds[1:]):
        t0 = float(times[a]); t1 = float(times[b]) if b < len(times) else total
        rows.append({"start": round(t0, 3), "end": round(t1, 3), "duration": round(t1 - t0, 3),
                     "state": "SPEECH" if sp[a] else "SILENCE"})
    return pd.DataFrame(rows), times, energy_db, thresh_db


def log_human_change(video_dir, user, segment_id, field, old_val, new_val, reason=""):
    """Append one audit-trail row (who / what / when) to changes.csv."""
    csv_path = Path(video_dir) / "changes.csv"
    row = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "user": user or "unknown", "segment_id": str(segment_id),
           "field": field, "old_value": str(old_val), "new_value": str(new_val), "reason": reason}
    pd.DataFrame([row]).to_csv(csv_path, mode="a" if csv_path.exists() else "w", header=not csv_path.exists(), index=False)

print("✅ Energy / VAD module ready.")


In [ ]:
# 🧠 Cell 5: AST Classification, Regions, Boundary Snapping, Region Editing Operations

SPEECH_NAMES = {"speech", "male speech, man speaking", "female speech, woman speaking",
                "child speech, kid speaking", "conversation", "narration, monologue"}
MUSIC_NAMES = {"music", "musical instrument", "singing", "song"}
SILENCE_NAMES = {"silence"}


class ASTInferenceEngine:
    """Pre-trained Audio Spectrogram Transformer. AudioSet class ids are resolved from the model's own id2label."""
    def __init__(self, model_name=None, device=None, allow_fallback=False):
        self.device = device or ("cuda" if (torch is not None and torch.cuda.is_available()) else "cpu")
        self.model_id = model_name or "MIT/ast-finetuned-audioset-10-10-0.4593"
        self.is_neural = False
        try:
            from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
            print(f"[AST] Loading '{self.model_id}' on {self.device.upper()}...")
            self.feature_extractor = AutoFeatureExtractor.from_pretrained(self.model_id)
            self.model = AutoModelForAudioClassification.from_pretrained(self.model_id).to(self.device).eval()
            id2label = {int(k): str(v).lower() for k, v in self.model.config.id2label.items()}
            self.speech_ids = [i for i, l in id2label.items() if l in SPEECH_NAMES]
            self.music_ids = [i for i, l in id2label.items() if l in MUSIC_NAMES]
            self.silence_ids = [i for i, l in id2label.items() if l in SILENCE_NAMES]
            if not (self.speech_ids and self.music_ids and self.silence_ids):
                raise RuntimeError("Could not find Speech/Music/Silence classes in model.config.id2label")
            print(f"[AST] class ids -> speech:{self.speech_ids} music:{self.music_ids} silence:{self.silence_ids}")
            self.is_neural = True
        except Exception as e:
            if not allow_fallback:
                raise RuntimeError(f"AST failed to load ({e}). Refusing to silently use a heuristic. "
                                   f"Fix the install/network or set ALLOW_HEURISTIC_FALLBACK=True for a demo only.") from e
            print("🚨🚨 AST NOT LOADED: using RMS/ZCR heuristic. DO NOT USE THESE LABELS FOR THE REAL CORPUS. 🚨🚨")

    def predict_scores(self, chunks, sr=DEFAULT_SR) -> np.ndarray:
        """Returns array (N, 3): [p_speech, p_music, p_silence]."""
        if not self.is_neural:
            out = []
            for ch in chunks:
                rms = float(np.sqrt(np.mean(ch ** 2)))
                zcr = float(np.mean(np.abs(np.diff(np.sign(ch)))) / 2.0)
                if rms < 0.005: out.append([0.05, 0.05, 0.9])
                elif zcr > 0.15: out.append([0.85, 0.05, 0.02])
                else: out.append([0.1, 0.75, 0.03])
            return np.array(out)
        inputs = self.feature_extractor(chunks, sampling_rate=sr, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.sigmoid(self.model(**inputs).logits).cpu().numpy()
        return np.stack([probs[:, self.speech_ids].max(1), probs[:, self.music_ids].max(1),
                         probs[:, self.silence_ids].max(1)], axis=1)


_AST_CACHE = {}

def get_ast_engine(device=None):
    if "e" not in _AST_CACHE:
        _AST_CACHE["e"] = ASTInferenceEngine(device=device, allow_fallback=ALLOW_HEURISTIC_FALLBACK)
    return _AST_CACHE["e"]


def decide_window(p_sp, p_mu, p_si, rms_db, floor_db):
    """Rule-based window label. Silence is decided by ENERGY, 'mixed' = speech AND music both present.
    Thresholds MIXED_SP_THR / MIXED_MU_THR should be tuned on the hand-labelled 10 minutes."""
    p_mix = float(np.sqrt(p_sp * p_mu))
    scores = {"speech": p_sp, "music": p_mu, "mixed": p_mix, "silence": p_si}
    sil_thr = floor_db + SILENCE_DB_ABOVE_FLOOR
    if rms_db < sil_thr:
        lbl = "silence"; scores["silence"] = max(p_si, 0.9)
        cert = float(np.clip((sil_thr - rms_db) / SILENCE_DB_ABOVE_FLOOR, 0, 1))
    elif p_sp >= MIXED_SP_THR and p_mu >= MIXED_MU_THR:
        lbl = "mixed"
        cert = float(np.clip(min(p_sp / MIXED_SP_THR, p_mu / MIXED_MU_THR) - 1.0, 0, 1))
    elif p_sp >= p_mu:
        lbl = "speech"; cert = float(np.clip(p_sp - p_mu, 0, 1))
    else:
        lbl = "music"; cert = float(np.clip(p_mu - p_sp, 0, 1))
    urg = round(1.0 - cert, 3)
    return {"pred_content": lbl, "confidence": round(float(np.clip(scores[lbl], 0, 1)), 3),
            "certainty": round(cert, 3), "urgency": urg,
            "priority": "URGENT" if urg > 0.65 else ("MEDIUM" if urg > 0.35 else "LOW"), "scores": scores}


def ast_predict_audio_windows(audio, sr=DEFAULT_SR, win_s=AST_WIN_S, hop_s=AST_HOP_S, batch_size=32,
                              device=None, floor_db=None, times=None, energy_db=None):
    engine = get_ast_engine(device)
    if energy_db is None:
        times, energy_db = compute_energy_db(audio, sr)
    if floor_db is None:
        floor_db, _ = adaptive_silence_floor(energy_db)
    total = len(audio)
    win_len, hop_len = int(win_s * sr), int(hop_s * sr)
    starts = list(range(0, max(1, total - hop_len // 2), hop_len))
    chunks, meta = [], []
    for s in starts:
        ch = audio[s:s + win_len]
        if len(ch) < win_len:
            ch = np.pad(ch, (0, win_len - len(ch)))
        chunks.append(ch)
        meta.append((s / sr, min(total, s + win_len) / sr))
    scores = np.concatenate([engine.predict_scores(chunks[b:b + batch_size], sr)
                             for b in range(0, len(chunks), batch_size)], axis=0)
    rows = []
    for (st, et), (p_sp, p_mu, p_si) in zip(meta, scores):
        k0, k1 = int(st / 0.010), max(int(st / 0.010) + 1, int(et / 0.010))
        seg = energy_db[k0:k1]
        rms_db = float(seg.mean()) if len(seg) else float(floor_db)
        d = decide_window(float(p_sp), float(p_mu), float(p_si), rms_db, floor_db)
        rows.append({"start": round(st, 3), "end": round(et, 3), "duration": round(et - st, 3),
                     "p_speech": round(float(p_sp), 3), "p_music": round(float(p_mu), 3),
                     "p_mixed": round(d["scores"]["mixed"], 3), "p_silence": round(float(p_si), 3),
                     "rms_db": round(rms_db, 2), "pred_content": d["pred_content"], "confidence": d["confidence"],
                     "certainty": d["certainty"], "urgency": d["urgency"], "review_priority": d["priority"]})
    return pd.DataFrame(rows)


def mode_filter(arr, k=3):
    out = arr.copy(); half = k // 2
    for i in range(len(arr)):
        vals, counts = np.unique(arr[max(0, i - half):min(len(arr), i + half + 1)], return_counts=True)
        out[i] = vals[np.argmax(counts)]
    return out


def finalize_regions(df):
    df = df.sort_values("start").reset_index(drop=True).copy()
    for c, dv in (("confidence", 0.8), ("urgency", 0.2)):
        if c not in df: df[c] = dv
    if "status" not in df: df["status"] = "unreviewed"
    df["start"], df["end"] = df["start"].round(3), df["end"].round(3)
    df["duration"] = (df["end"] - df["start"]).round(3)
    df["review_priority"] = np.where(df["urgency"] > 0.65, "URGENT", np.where(df["urgency"] > 0.35, "MEDIUM", "LOW"))
    df["confidence"], df["urgency"] = df["confidence"].round(3), df["urgency"].round(3)
    return df[["start", "end", "duration", "label", "confidence", "urgency", "review_priority", "status"]
              + [c for c in df.columns if c not in ("start", "end", "duration", "label", "confidence", "urgency", "review_priority", "status")]]


def _merge_adjacent_same_label(df):
    rows = []
    for r in df.sort_values("start").to_dict("records"):
        if rows and rows[-1]["label"] == r["label"]:
            a = rows[-1]; da, db = a["end"] - a["start"], r["end"] - r["start"]; tot = max(da + db, 1e-9)
            for k in ("confidence", "urgency"):
                a[k] = (a.get(k, 0.5) * da + r.get(k, 0.5) * db) / tot
            a["end"] = r["end"]
            if r.get("status") == "human_edited": a["status"] = "human_edited"
        else:
            rows.append(dict(r))
    return finalize_regions(pd.DataFrame(rows))


def windows_to_regions(windows_df, labels_idx, total_dur, hop_s=AST_HOP_S):
    """Each window OWNS only [center - hop/2, center + hop/2]. Owned slices tile the file, so regions never overlap
    (v13 used the window ends, which overlapped by win-hop seconds)."""
    win = float(windows_df["duration"].iloc[0]) if len(windows_df) else AST_WIN_S
    own = []
    for i, r in windows_df.reset_index(drop=True).iterrows():
        c = r["start"] + win / 2.0
        own.append((0.0 if i == 0 else c - hop_s / 2.0,
                    total_dur if i == len(windows_df) - 1 else min(total_dur, c + hop_s / 2.0)))
    rows, a = [], 0
    for i in range(1, len(labels_idx) + 1):
        if i == len(labels_idx) or labels_idx[i] != labels_idx[a]:
            sub = windows_df.iloc[a:i]
            s, e = own[a][0], own[i - 1][1]
            if e - s > 1e-3:
                urg = float(sub["urgency"].mean())
                rows.append({"start": s, "end": e, "label": CONTENT_LABELS[labels_idx[a]],
                             "confidence": float(sub["confidence"].mean()), "urgency": urg, "status": "unreviewed"})
            a = i
    return finalize_regions(pd.DataFrame(rows))


def merge_short_regions(df, min_s=MIN_REGION_S):
    df = df.reset_index(drop=True).copy()
    while len(df) > 1:
        d = df["end"] - df["start"]
        short = d[d < min_s]
        if short.empty:
            break
        i = int(short.idxmin())
        j = 1 if i == 0 else (i - 1 if i == len(df) - 1 else (i - 1 if d[i - 1] >= d[i + 1] else i + 1))
        if j < i: df.loc[j, "end"] = df.loc[i, "end"]
        else: df.loc[j, "start"] = df.loc[i, "start"]
        df = df.drop(index=i).reset_index(drop=True)
    return _merge_adjacent_same_label(df)


def refine_region_boundaries(df, pauses_df, snap_s=SNAP_WINDOW_S):
    """AST borders are only accurate to ~+-hop/2. Snap each border to the nearest VAD pause:
    speech->non-speech: just after speech ends (pause start); non-speech->speech: just before it starts (pause end);
    other transitions: pause centre. Order and a 0.5 s minimum region length are preserved."""
    df = df.sort_values("start").reset_index(drop=True).copy()
    if len(df) < 2 or pauses_df is None or pauses_df.empty:
        df["refined"] = True; return df
    P = pauses_df[pauses_df["duration"] >= MIN_PAUSE_CAND]
    orig = list(df["end"].values[:-1]); new = list(orig)
    for i in range(len(orig)):
        b = orig[i]; a_l, b_l = df.loc[i, "label"], df.loc[i + 1, "label"]
        near = P[(P["end"] >= b - snap_s) & (P["start"] <= b + snap_s)]
        if near.empty: continue
        sp = ("speech", "mixed")
        if a_l in sp and b_l not in sp: pts = near["start"].values + np.minimum(0.15, near["duration"].values / 2)
        elif a_l not in sp and b_l in sp: pts = near["end"].values - np.minimum(0.15, near["duration"].values / 2)
        else: pts = near["center"].values
        cand = float(pts[int(np.argmin(np.abs(pts - b)))])
        lo = (new[i - 1] if i > 0 else df.loc[0, "start"]) + 0.5
        hi = (orig[i + 1] if i + 1 < len(orig) else df.loc[len(df) - 1, "end"]) - 0.5
        if lo <= cand <= hi: new[i] = cand
    for i, nb in enumerate(new):
        df.loc[i, "end"] = nb; df.loc[i + 1, "start"] = nb
    df["refined"] = True
    return finalize_regions(df.assign(refined=True))


def classify_audio(y, sr=DEFAULT_SR, win_s=AST_WIN_S, hop_s=AST_HOP_S, batch_size=32, device=None, pauses_df=None):
    """AST windows -> energy-based silence -> mode filter -> non-overlapping regions -> merge short -> snap to pauses."""
    times, energy_db = compute_energy_db(y, sr)
    floor_db, _ = adaptive_silence_floor(energy_db)
    windows_df = ast_predict_audio_windows(y, sr, win_s, hop_s, batch_size, device, floor_db, times, energy_db)
    if windows_df.empty:
        return pd.DataFrame(), windows_df
    idx = np.array([CONTENT_LABELS.index(l) for l in windows_df["pred_content"]])
    regions = windows_to_regions(windows_df, mode_filter(idx, 3), len(y) / sr, hop_s)
    regions = merge_short_regions(regions, MIN_REGION_S)
    if pauses_df is not None:
        regions = merge_short_regions(refine_region_boundaries(regions, pauses_df), MIN_REGION_S)
    return regions, windows_df


# ---- pure region-editing operations (used by the spectrogram editor; unit-testable) -----------------
def _renorm(df):
    df = df.sort_values("start").reset_index(drop=True).copy()
    df["duration"] = (df["end"] - df["start"]).round(3)
    return df

def region_set_label(df, idx, label):
    df = df.copy(); df.loc[idx, "label"] = label; df.loc[idx, "confidence"] = 1.0
    df.loc[idx, "urgency"] = 0.0; df.loc[idx, "status"] = "human_edited"
    return _renorm(df)

def region_move_borders(df, idx, new_start, new_end):
    df = df.copy(); n = len(df)
    if idx == 0: new_start = float(df.loc[0, "start"])            # file start is fixed
    if idx == n - 1: new_end = float(df.loc[n - 1, "end"])        # file end is fixed
    lo = df.loc[idx - 1, "start"] + 0.05 if idx > 0 else 0.0
    hi = df.loc[idx + 1, "end"] - 0.05 if idx < n - 1 else df.loc[idx, "end"]
    if not (lo <= new_start < new_end <= hi):
        raise ValueError(f"Invalid borders: need {lo:.2f} <= start < end <= {hi:.2f}")
    df.loc[idx, ["start", "end"]] = [new_start, new_end]; df.loc[idx, "status"] = "human_edited"
    if idx > 0: df.loc[idx - 1, "end"] = new_start; df.loc[idx - 1, "status"] = "human_edited"
    if idx < n - 1: df.loc[idx + 1, "start"] = new_end; df.loc[idx + 1, "status"] = "human_edited"
    return _renorm(df)

def region_split(df, idx, t):
    r = df.loc[idx]
    if not (r["start"] + 0.05 < t < r["end"] - 0.05): raise ValueError("Split time must be inside the region")
    a, b = r.copy(), r.copy(); a["end"] = t; b["start"] = t; a["status"] = b["status"] = "human_edited"
    return _renorm(pd.concat([df.drop(index=idx), pd.DataFrame([a, b])], ignore_index=True))

def region_merge_with_next(df, idx):
    if idx >= len(df) - 1: raise ValueError("No next region")
    df = df.copy(); df.loc[idx, "end"] = df.loc[idx + 1, "end"]; df.loc[idx, "status"] = "human_edited"
    return _renorm(df.drop(index=idx + 1))

def region_delete(df, idx):
    if len(df) < 2: raise ValueError("Cannot delete the only region")
    df = df.copy()
    if idx > 0: df.loc[idx - 1, "end"] = df.loc[idx, "end"]; df.loc[idx - 1, "status"] = "human_edited"
    else: df.loc[1, "start"] = df.loc[0, "start"]; df.loc[1, "status"] = "human_edited"
    return _renorm(df.drop(index=idx))

def region_paint(df, a, b, label):
    """Add/overwrite: relabel the interval [a, b] (splitting regions it overlaps)."""
    if b - a < 0.05: raise ValueError("Interval too short")
    rows = []
    for r in df.to_dict("records"):
        if r["end"] <= a or r["start"] >= b: rows.append(r); continue
        if r["start"] < a - 0.05: l = dict(r); l["end"] = a; rows.append(l)
        if r["end"] > b + 0.05: rr = dict(r); rr["start"] = b; rows.append(rr)
    rows.append({"start": a, "end": b, "label": label, "confidence": 1.0, "urgency": 0.0,
                 "review_priority": "LOW", "status": "human_edited"})
    return _renorm(pd.DataFrame(rows))

print("✅ AST / regions module ready.")

In [ ]:
# ✂️ Cell 6: Cut Engine v2 (Global DP, Short-Pause Handling, Edge Snapping, Word Check, Greedy Baseline)

ARABIC_STOPS = ("د", "ت", "ط", "ق", "ك", "ب", "ء", "أ", "إ", "ؤ", "ئ")


@dataclass
class Cand:
    t: float
    p_start: float
    p_end: float
    score: float
    weak: bool = False       # energy valley / word gap, not a true pause
    short: bool = False      # pause < 200 ms: cannot give 100 ms of margin on both sides
    word_ok: float = 0.5     # 1.0 verified by word timestamps, 0.5 unverified
    word_risk: bool = False  # next word starts with a plosive/glottal stop (closure can look like a pause)


def pause_score(p_start, p_end, times, energy_db, thr_db, quiet):
    dur = min(p_end - p_start, 1.0)
    k0, k1 = int(np.searchsorted(times, p_start)), int(np.searchsorted(times, p_end))
    if k1 <= k0:
        return round(0.5 * dur, 3)
    depth = float(np.clip((thr_db - float(energy_db[k0:k1].mean())) / 15.0, 0.0, 1.0))
    return round(float(np.clip(0.5 * dur + 0.25 * depth + 0.25 * float(quiet[k0:k1].mean()), 0.0, 1.0)), 3)


def cand_from_pause(p, times, energy_db, thr_db, quiet):
    sc = pause_score(p["start"], p["end"], times, energy_db, thr_db, quiet)
    return Cand(t=float(p["center"]), p_start=float(p["start"]), p_end=float(p["end"]), score=sc,
                short=(p["end"] - p["start"]) < MIN_PAUSE_STRONG - 1e-9)


def snap_edge_to_pause(t, pauses_df, side, lo, hi, back=3.0, fwd=0.5, min_dur=MIN_PAUSE_STRONG):
    """Stream edges (from coarse AST borders) are snapped to a real pause. 'start': prefer a pause BEFORE the edge
    (extends into non-speech); 'end': prefer a pause AFTER it. Never reaches into a neighbouring stream (lo/hi)."""
    sub = pauses_df[pauses_df["duration"] >= min_dur - 1e-9]
    if side == "start": sub = sub[(sub["end"] >= t - back) & (sub["start"] <= t + fwd)]
    else: sub = sub[(sub["end"] >= t - fwd) & (sub["start"] <= t + back)]
    sub = sub[(sub["end"] > lo) & (sub["start"] < hi)]
    if sub.empty:
        return None
    d = np.maximum(0.0, np.maximum(sub["start"].values - t, t - sub["end"].values))
    return sub.iloc[int(np.argmin(d))]


def build_streams(regions_df, gap=STREAM_MERGE_GAP, min_len=MIN_LEN, short_gap=SHORT_STREAM_GAP):
    """Merge speech/mixed regions into continuous streams; streams < 5 s are merged into a neighbour if <= 3 s away."""
    sm = regions_df[regions_df["label"].isin(["speech", "mixed"])].sort_values("start")
    streams = []
    for r in sm.itertuples():
        if streams and r.start <= streams[-1][1] + gap: streams[-1][1] = max(streams[-1][1], float(r.end))
        else: streams.append([float(r.start), float(r.end)])
    changed = True
    while changed:
        changed = False
        for i, (s, e) in enumerate(streams):
            if e - s < min_len:
                gl = s - streams[i - 1][1] if i > 0 else 1e9
                gr = streams[i + 1][0] - e if i + 1 < len(streams) else 1e9
                if min(gl, gr) <= short_gap:
                    if gl <= gr: streams[i - 1][1] = e
                    else: streams[i + 1][0] = s
                    del streams[i]; changed = True; break
    return [(s, e) for s, e in streams]


# ---- word-timestamp verification (faster-whisper, Arabic) ---------------------------------------------------
def _load_word_model(size="small"):
    try:
        from faster_whisper import WhisperModel
        dev = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"
        m = WhisperModel(size, device=dev, compute_type="float16" if dev == "cuda" else "int8")
        print(f"🎙️ faster-whisper '{size}' loaded for word-boundary verification ({dev}).")
        return m
    except Exception as e:
        print(f"ℹ️ faster-whisper unavailable ({e}). Word check skipped: word_ok stays 0.5 (unverified).")
        return None


def transcribe_words(model, audio_slice, offset):
    segs, _ = model.transcribe(np.ascontiguousarray(audio_slice, dtype=np.float32), language="ar",
                               word_timestamps=True, beam_size=1, condition_on_previous_text=False)
    return [{"start": round(w.start + offset, 3), "end": round(w.end + offset, 3), "word": w.word}
            for s in segs for w in (s.words or [])]


def verify_candidates_with_words(cands, words, tol=WORD_TS_TOL):
    """Verify candidate cuts against Whisper word timestamps.
    Note: faster-whisper stretches word end timestamps across acoustic silences to the next word start.
    Therefore, candidate pauses confirmed by acoustic energy/VAD are valid inter-word boundaries.
    Only reject candidate cuts if they are weak valley cuts landing deep inside active phonation of a single short word.
    """
    if not words:
        return cands, 0, 0
    starts = [w["start"] for w in words]
    kept, rejected, verified = [], 0, 0
    for c in cands:
        i = bisect.bisect_right(starts, c.t)
        prev_w = words[i - 1] if i > 0 else None
        next_w = words[i] if i < len(words) else None

        if c.weak and prev_w:
            w_len = prev_w["end"] - prev_w["start"]
            if w_len < 0.60 and prev_w["start"] + 0.08 < c.t < prev_w["end"] - 0.08:
                rejected += 1
                continue

        c.word_ok = 1.0
        verified += 1
        if next_w and c.short and 0.0 <= next_w["start"] - c.p_end <= 0.10 and next_w["word"].strip().startswith(ARABIC_STOPS):
            c.word_risk = True
        kept.append(c)
    return kept, rejected, verified


def make_valley_cand(vt, t0, t1, times, energy_db, thr_db, words):
    """Forced candidate inside a long pause-free stretch. With word timestamps we pick the quietest GAP BETWEEN WORDS
    (never inside a word); without them, the energy minimum within +-1 s."""
    if words:
        best = None
        for k in range(len(words) - 1):
            g0, g1 = words[k]["end"], words[k + 1]["start"]
            if g1 - g0 >= 0.02 and abs((g0 + g1) / 2 - vt) <= 1.5 and t0 < (g0 + g1) / 2 < t1:
                k0, k1 = int(np.searchsorted(times, g0)), max(int(np.searchsorted(times, g1)), int(np.searchsorted(times, g0)) + 1)
                m = float(energy_db[k0:k1].mean())
                if best is None or m < best[0]: best = (m, g0, g1)
        if best:
            m, g0, g1 = best
            return Cand(t=round((g0 + g1) / 2, 3), p_start=g0, p_end=g1, weak=True, word_ok=1.0,
                        score=round(max(0.05, min(0.35, (thr_db + 7.0 - m) / 20.0)), 3))
    k0, k1 = max(0, int((vt - 1.0) / 0.010)), min(len(energy_db), int((vt + 1.0) / 0.010))
    if k1 <= k0:
        return None
    j = k0 + int(np.argmin(energy_db[k0:k1])); mt = round(float(times[j]), 3)
    if not (t0 < mt < t1):
        return None
    return Cand(t=mt, p_start=max(t0, mt - 0.08), p_end=min(t1, mt + 0.08), weak=True,
                score=round(max(0.05, min(0.35, (thr_db + 7.0 - float(energy_db[j])) / 20.0)), 3))


# ---- global shortest-path DP -----------------------------------------------------------------------------
def _node_start(cand, raw_t):
    return raw_t if (cand is None or cand.weak) else max(raw_t, cand.p_end - MARGIN_MAX)

def _node_end(cand, raw_t):
    return raw_t if (cand is None or cand.weak) else min(raw_t, cand.p_start + MARGIN_MAX)

def _dp(nodes, lo, mid, hi):
    n, INF = len(nodes), 1e18
    cost, prev = [INF] * n, [-1] * n
    cost[0] = 0.0
    for j in range(1, n):
        last = (j == n - 1)
        e_j = _node_end(nodes[j][1], nodes[j][0])
        for i in range(j - 1, -1, -1):
            L = e_j - _node_start(nodes[i][1], nodes[i][0])
            if L <= 0: continue
            if L > hi: break
            if L < lo or cost[i] >= INF: continue
            c = cost[i] + LAM * ((L - TARGET) / TARGET) ** 2
            if L > mid: c += FB_PEN
            if not last and nodes[j][1] is not None:
                cd = nodes[j][1]
                c += CUT_W * (1.0 - cd.score) + (WEAK_PEN if cd.weak else 0.0) + (SHORT_PEN if cd.short else 0.0)
            if c < cost[j]: cost[j], prev[j] = c, i
    if cost[-1] >= INF:
        return None
    path, j = [], n - 1
    while j > 0: path.append(j); j = prev[j]
    return path[::-1]


def plan_cuts(t0, t1, cands, c0=None, c1=None):
    """Returns ([(start, end, start_cand, end_cand, used_fallback)], plan_tag)."""
    if _node_end(c1, t1) - _node_start(c0, t0) < MIN_LEN:
        return [(_node_start(c0, t0), _node_end(c1, t1), c0, c1, False)], "short_stream"
    cs = sorted((c for c in cands if t0 < c.t < t1), key=lambda c: c.t)
    nodes = [(t0, c0)] + [(c.t, c) for c in cs] + [(t1, c1)]
    for lo, mid, hi, tag in [(MIN_LEN, MAX_LEN, FB_LEN, "ok"), (4.0, MAX_LEN, 20.0, "relaxed")]:
        path = _dp(nodes, lo, mid, hi)
        if path:
            segs, a = [], 0
            for j in path:
                s, e = _node_start(nodes[a][1], nodes[a][0]), _node_end(nodes[j][1], nodes[j][0])
                segs.append((s, e, nodes[a][1], nodes[j][1], bool(e - s > MAX_LEN)))
                a = j
            return segs, tag
    s0, e1 = _node_start(c0, t0), _node_end(c1, t1)
    k = max(1, int(np.ceil((e1 - s0) / 12.0)))
    ed = np.linspace(s0, e1, k + 1)
    return [(float(ed[i]), float(ed[i + 1]), c0 if i == 0 else None, c1 if i == k - 1 else None, True) for i in range(k)], "equal_split"


def greedy_plan(t0, t1, cands, c0=None, c1=None):
    """Baseline for the ablation: walk left to right, take the best-scoring pause that gives a 5-15 s segment."""
    cs = sorted((c for c in cands if t0 < c.t < t1), key=lambda c: c.t)
    end_all, segs = _node_end(c1, t1), []
    cur_t, cur_c = t0, c0
    while True:
        s = _node_start(cur_c, cur_t)
        if end_all - s <= MAX_LEN + 1e-6:
            segs.append((s, end_all, cur_c, c1, False)); break
        ok = [c for c in cs if c.t > cur_t and MIN_LEN <= _node_end(c, c.t) - s <= MAX_LEN]
        fbk = [c for c in cs if c.t > cur_t and MAX_LEN < _node_end(c, c.t) - s <= FB_LEN]
        if ok: c = max(ok, key=lambda x: x.score)
        elif fbk: c = min(fbk, key=lambda x: _node_end(x, x.t) - s)
        else: c = None
        if c is None:
            e = s + TARGET; segs.append((s, e, cur_c, None, True)); cur_t, cur_c = e, None
        else:
            e = _node_end(c, c.t); segs.append((s, e, cur_c, c, e - s > MAX_LEN)); cur_t, cur_c = c.t, c
    return segs


# ---- certainty, margins, fade --------------------------------------------------------------------------
def cut_certainty(cand, used_fallback, plan_tag, is_file_edge=False):
    if cand is None:
        s = 0.95 if is_file_edge else 0.0
    else:
        s = 0.40 * min(cand.score / 0.8, 1.0) + 0.25 * cand.word_ok + 0.35 * (0.0 if cand.weak else 1.0)
        if cand.word_risk: s -= 0.10
        if cand.weak: s = min(s, 0.40)
        if cand.short: s = min(s, 0.60)
    if used_fallback: s = min(s, 0.60)
    if plan_tag != "ok": s = min(s, 0.45)
    return round(float(np.clip(s, 0.0, 1.0)), 3)


def get_tier(cert):
    return "HIGH" if cert >= HIGH_TIER else ("MEDIUM" if cert >= MED_TIER else "LOW")


def segment_margins(seg_s, seg_e, cand_s, cand_e):
    """(lead_ms, trail_ms, margin_ok). Real pause cuts must keep 100-200 ms of silence; weak cuts have 0 -> not ok."""
    lead = int(round((cand_s.p_end - seg_s) * 1000)) if (cand_s and not cand_s.weak) else 0
    trail = int(round((seg_e - cand_e.p_start) * 1000)) if (cand_e and not cand_e.weak) else 0
    lo, hi = MARGIN_MIN * 1000 - MARGIN_TOL_MS, MARGIN_MAX * 1000 + MARGIN_TOL_MS
    ok = (cand_s is None or lo <= lead <= hi) and (cand_e is None or lo <= trail <= hi)
    return lead, trail, bool(ok)


def apply_hann_fade(x, sr=DEFAULT_SR, fade_ms=8.0):
    n = int(sr * fade_ms / 1000.0)
    if n <= 0 or len(x) < 2 * n: return x
    out = x.copy()
    out[:n] *= 0.5 * (1.0 - np.cos(np.linspace(0, np.pi, n)))
    out[-n:] *= 0.5 * (1.0 + np.cos(np.linspace(0, np.pi, n)))
    return out


def region_label_for(s, e, regions_df):
    """Segment label = region label with the largest overlap (so human overrides in the region editor are respected)."""
    ov = {}
    for r in regions_df.itertuples():
        o = min(e, r.end) - max(s, r.start)
        if o > 0:
            a = ov.setdefault(r.label, [0.0, 0.0]); a[0] += o; a[1] += o * float(getattr(r, "confidence", 0.8))
    if not ov: return "speech", 0.5
    pool = {k: v for k, v in ov.items() if k != "silence"} or ov
    best = max(pool, key=lambda k: pool[k][0])
    return best, round(pool[best][1] / pool[best][0], 3)


def _new_stats():
    return {"n": 0, "dur": 0.0, "in_5_15": 0, "over_15": 0, "under_5": 0, "margin_ok": 0, "blind_cuts": 0, "weak_cuts": 0}

def _plan_stats(segs, st):
    for i, (s, e, sc, ec, fb) in enumerate(segs):
        d = e - s
        st["n"] += 1; st["dur"] += d
        st["in_5_15"] += int(MIN_LEN - 0.01 <= d <= MAX_LEN + 0.01)
        st["over_15"] += int(d > MAX_LEN + 0.01); st["under_5"] += int(d < MIN_LEN - 0.01)
        st["margin_ok"] += int(segment_margins(s, e, sc, ec)[2])
        if i < len(segs) - 1:
            st["blind_cuts"] += int(ec is None); st["weak_cuts"] += int(ec is not None and ec.weak)


def cut_speech_and_mixed_segments(audio, regions_df, out_dir, sr=DEFAULT_SR, pauses=None,
                                  use_word_verification=True, word_model_size="small", run_ablation=True, **_ignored):
    """Cut speech/mixed streams into 5-15 s segments at real pauses; keep music regions whole. Writes
    segments.csv, review_queue_urgent.csv, ablation_metrics.json, word_timestamps.json and segments/*.wav."""
    out_dir = Path(out_dir); seg_dir = out_dir / "segments"; seg_dir.mkdir(parents=True, exist_ok=True)
    for f in seg_dir.glob("segment_*.wav"): f.unlink()
    total_dur = len(audio) / sr

    reg_csv = out_dir / "regions.csv"
    if reg_csv.exists():
        regions_df = pd.read_csv(reg_csv); print("ℹ️ Using regions.csv (includes any human edits).")
    regions_df = regions_df.sort_values("start").reset_index(drop=True)

    times, energy_db = compute_energy_db(audio, sr)
    floor_db, thr_db = adaptive_silence_floor(energy_db)
    quiet = energy_db < thr_db

    if pauses is None or not isinstance(pauses, pd.DataFrame):
        pauses_df = detect_speech_activity_overlay(audio, sr)[1]
    else:
        pauses_df = pauses.copy()
        if len(pauses_df) < 800 and total_dur > 1800:
            energy_pauses = detect_pauses_energy(audio, sr, min_pause_ms=120)
            existing_starts = pauses_df["start"].to_numpy()
            existing_ends = pauses_df["end"].to_numpy()
            extra_pauses = []
            for ep in energy_pauses:
                if not np.any((existing_starts < ep.end) & (existing_ends > ep.start)):
                    extra_pauses.append({
                        "pause_id": f"PE{len(pauses_df) + len(extra_pauses) + 1:04d}",
                        "start": ep.start, "end": ep.end, "center": ep.center,
                        "duration": ep.duration, "duration_ms": ep.duration_ms,
                        "is_safe": ep.duration >= 0.250
                    })
            if extra_pauses:
                pauses_df = pd.concat([pauses_df, pd.DataFrame(extra_pauses)], ignore_index=True)
                pauses_df = pauses_df.sort_values("start").reset_index(drop=True)
    floor_db, thr_db = adaptive_silence_floor(energy_db)
    quiet = energy_db < thr_db

    word_model = _load_word_model(word_model_size) if use_word_verification else None
    streams = build_streams(regions_df)
    pieces, all_words = [], []
    abl = {"dp": _new_stats(), "greedy": _new_stats()}
    n_rejected = n_verified = 0

    for si, (st_s, st_e) in enumerate(streams):
        lo_b = streams[si - 1][1] if si > 0 else 0.0
        hi_b = streams[si + 1][0] if si + 1 < len(streams) else total_dur
        c0 = c1 = None; t0, t1 = st_s, st_e
        if st_s > 0.05:
            p = snap_edge_to_pause(st_s, pauses_df, "start", lo_b, st_e)
            if p is not None: c0 = cand_from_pause(p, times, energy_db, thr_db, quiet); t0 = max(c0.t, lo_b)
        if st_e < total_dur - 0.05:
            p = snap_edge_to_pause(st_e, pauses_df, "end", st_s, hi_b)
            if p is not None: c1 = cand_from_pause(p, times, energy_db, thr_db, quiet); t1 = min(c1.t, hi_b)
        if t1 - t0 < 1.0:
            continue

        words = []
        if word_model is not None:
            try:
                words = transcribe_words(word_model, audio[int(max(0, t0) * sr):int(min(total_dur, t1) * sr)], max(0.0, t0))
                all_words.extend(words)
            except Exception as e:
                print(f"⚠️ Whisper failed on stream {si}: {e}")

        cands = []
        for p in pauses_df.itertuples():
            if p.duration >= MIN_PAUSE_CAND - 1e-9 and t0 < p.center < t1:
                cands.append(cand_from_pause({"start": p.start, "end": p.end, "center": p.center}, times, energy_db, thr_db, quiet))
        cands, rj, vf = verify_candidates_with_words(cands, words)
        n_rejected += rj; n_verified += vf

        anchors = [t0] + [c.t for c in cands] + [t1]
        extra = []
        for a, b in zip(anchors[:-1], anchors[1:]):
            if b - a > 12.0:
                n_v = int(np.ceil((b - a) / 10.0)) - 1
                for vi in range(1, n_v + 1):
                    v = make_valley_cand(a + vi * (b - a) / (n_v + 1), t0, t1, times, energy_db, thr_db, words)
                    if v: extra.append(v)
        cands = sorted(cands + extra, key=lambda c: c.t)

        segs, tag = plan_cuts(t0, t1, cands, c0, c1)
        if run_ablation and tag == "ok":
            _plan_stats(segs, abl["dp"]); _plan_stats(greedy_plan(t0, t1, cands, c0, c1), abl["greedy"])

        for (s, e, sc, ec, fb) in segs:
            if e - s < 1.0: continue
            pieces.append({"kind": "cut", "s": s, "e": e, "sc": sc, "ec": ec, "fb": fb, "tag": tag})

    # music regions are kept whole (never cut); trimmed so they never overlap a speech/mixed segment
    cut_pieces = [p for p in pieces if p["kind"] == "cut"]
    for r in regions_df[regions_df["label"] == "music"].itertuples():
        ms, me = float(r.start), float(r.end)
        for sp in cut_pieces:
            if sp["e"] > ms and sp["s"] < me:
                if sp["s"] <= ms: ms = max(ms, sp["e"])
                else: me = min(me, sp["s"])
        if me - ms >= 2.0:
            pieces.append({"kind": "music", "s": ms, "e": me, "conf": float(getattr(r, "confidence", 0.8)),
                           "urg": float(getattr(r, "urgency", 0.2))})
    pieces.sort(key=lambda p: p["s"])

    rows = []
    for k, p in enumerate(pieces, 1):
        s, e = p["s"], p["e"]
        sid = f"{k:04d}"; wav_name = f"segment_{sid}.wav"
        i0, i1 = max(0, int(s * sr)), min(len(audio), int(e * sr))
        write_wav(seg_dir / wav_name, apply_hann_fade(audio[i0:i1].copy(), sr), sr)
        base = {"video_id": out_dir.name, "segment_id": sid, "wav_file": f"segments/{wav_name}",
                "start": round(s, 3), "end": round(e, 3), "duration": round(e - s, 3)}
        if p["kind"] == "music":
            rows.append({**base, "label": "music", "label_conf": round(p["conf"], 3), "cut_type": "music_whole",
                         "trail_ms": 0, "lead_ms": 0, "needs_review": int(p["urg"] > 0.65), "pause_ms": 0, "score": 0.0,
                         "cut_certainty": 1.0, "cut_tier": "HIGH", "word_check_passed": "NA", "fallback_used": 0,
                         "urgency": round(p["urg"], 3), "review_priority": "URGENT" if p["urg"] > 0.65 else "LOW",
                         "status": "pending_student2", "margin_ok": 1, "plan_tag": "music"})
            continue
        sc, ec, fb, tag = p["sc"], p["ec"], p["fb"], p["tag"]
        lead, trail, m_ok = segment_margins(s, e, sc, ec)
        cert = min(cut_certainty(sc, fb, tag, s <= 0.05), cut_certainty(ec, fb, tag, e >= total_dur - 0.05))
        tier = get_tier(cert)
        lbl, lconf = region_label_for(s, e, regions_df)
        urg = round(float(np.clip(0.35 * (1 - lconf) + 0.35 * (1 - cert) + 0.15 * float(fb) + 0.15 * float(not m_ok), 0, 1)), 3)
        needs = int(tier == "LOW" or fb or (tier == "MEDIUM" and urg >= 0.30))
        pri = "URGENT" if (tier == "LOW" or fb) else ("MEDIUM" if needs else "LOW")
        if tag == "short_stream": ctype = "short_stream"
        elif ec is None: ctype = "file_edge" if e >= total_dur - 0.05 else "split"
        elif ec.weak: ctype = "valley"
        elif fb: ctype = "fallback"
        elif ec.short: ctype = "short_pause"
        else: ctype = "pause"
        lbl, lconf = region_label_for(s, e, regions_df)
        urg = round(float(np.clip(0.35 * (1 - lconf) + 0.35 * (1 - cert) + 0.15 * float(fb) + 0.15 * float(not m_ok), 0, 1)), 3)
        rows.append({**base, "label": lbl, "label_conf": lconf, "cut_type": ctype, "trail_ms": trail, "lead_ms": lead,
                     "needs_review": needs, "pause_ms": int(round((ec.p_end - ec.p_start) * 1000)) if ec else 0,
                     "score": round(ec.score, 3) if ec else 0.0, "cut_certainty": cert, "cut_tier": tier,
                     "word_check_passed": "1" if (ec and ec.word_ok == 1.0) else "NA", "fallback_used": int(fb),
                     "urgency": urg, "review_priority": pri,
                     "status": "pending_student2", "margin_ok": int(m_ok), "plan_tag": tag})

    segments_df = pd.DataFrame(rows)
    segments_df.to_csv(out_dir / "segments.csv", index=False, encoding="utf-8")
    segments_df.sort_values(["needs_review", "cut_certainty", "urgency"], ascending=[False, True, False]).to_csv(out_dir / "review_queue_urgent.csv", index=False, encoding="utf-8")
    (out_dir / "ablation_metrics.json").write_text(json.dumps(abl, indent=2), encoding="utf-8")
    if all_words: (out_dir / "word_timestamps.json").write_text(json.dumps(all_words, ensure_ascii=False), encoding="utf-8")

    cutdf = segments_df[segments_df["cut_type"] != "music_whole"]
    n = max(1, len(cutdf))
    print("=" * 66); print("🎯 CUT ENGINE V2 — MEASURED METRICS (speech/mixed segments)"); print("=" * 66)
    print(f"• Streams                      : {len(streams)}   | silence floor {floor_db:.1f} dB, threshold {thr_db:.1f} dB")
    print(f"• Segments (speech+mixed)      : {len(cutdf)}   | music rows: {len(segments_df) - len(cutdf)}")
    print(f"• In 5-15 s window             : {int(((cutdf.duration >= 4.99) & (cutdf.duration <= 15.01)).sum())} / {len(cutdf)}")
    print(f"• Fallback (15-18 s)           : {int(cutdf.fallback_used.sum())}   | short_stream/relaxed/split: {int((cutdf.plan_tag != 'ok').sum())}")
    print(f"• Margin 100-200 ms satisfied  : {100 * cutdf.margin_ok.mean():.1f}%" if len(cutdf) else "")
    print(f"• Whisper word check           : verified {n_verified}, rejected {n_rejected}")
    print(f"• Review workload              : {int(cutdf.needs_review.sum())} / {len(cutdf)} ({100 * cutdf.needs_review.sum() / n:.1f}%)  [target 5-15%]")
    print(f"• Saved                        : {out_dir / 'segments.csv'}"); print("=" * 66)
    return segments_df

print("✅ Cut Engine v2 ready.")


In [ ]:
# 🔍 Cell 7: Interactive Inspectors

LABEL_COLORS = {"speech": "#2ca02c", "music": "#1f77b4", "mixed": "#ff7f0e", "silence": "#7f7f7f"}


def plot_spectrogram_with_regions(audio, regions_df, t0, t1, sr=DEFAULT_SR, selected=None, title=""):
    """Linear-frequency spectrogram (0-8 kHz, 10 ms hop) with coloured region overlays and region borders."""
    t0, t1 = max(0.0, t0), min(len(audio) / sr, t1)
    seg = audio[int(t0 * sr):int(t1 * sr)]
    f, tt, S = signal.spectrogram(seg, fs=sr, nperseg=400, noverlap=240, nfft=512, mode="psd")
    S_db = 10 * np.log10(S + 1e-12)
    fig, ax = plt.subplots(figsize=(13, 4.2))
    ax.imshow(S_db, origin="lower", aspect="auto", extent=[t0, t0 + S_db.shape[1] * 0.010, 0, sr / 2000.0],
              vmin=S_db.max() - 80, vmax=S_db.max(), cmap="magma")
    for i, r in regions_df.iterrows():
        if r["end"] < t0 or r["start"] > t1: continue
        a, b = max(t0, r["start"]), min(t1, r["end"])
        ax.axvspan(a, b, color=LABEL_COLORS.get(r["label"], "w"), alpha=0.28 if i != selected else 0.45)
        ax.axvline(r["start"], color="white", ls="--", lw=1.0); ax.axvline(r["end"], color="white", ls="--", lw=1.0)
        ax.text((a + b) / 2, sr / 2000.0 * 0.93, f"#{i} {r['label']}", color="white", ha="center", fontsize=9, fontweight="bold")
    ax.set_xlim(t0, t1); ax.set_xlabel("Time (s)"); ax.set_ylabel("Frequency (kHz)"); ax.set_title(title or "Spectrogram + regions")
    plt.tight_layout()
    return fig


def build_synchronized_player_iframe(audio_samples, sr, t_val, win_dur, intervals_df, times_arr, energy_db_arr, thresh_db, total_audio_dur):
    """Self-contained HTML5 player with a cursor synchronised to playback (inside an iframe)."""
    import html
    win_s = max(0.0, t_val - (win_dur / 2.0)); win_e = min(total_audio_dur, win_s + win_dur)
    if (win_e - win_s) < win_dur and win_s > 0: win_s = max(0.0, win_e - win_dur)
    actual_dur = win_e - win_s
    snippet = audio_samples[int(win_s * sr):int(win_e * sr)]

    fig, (ax_wave, ax_energy) = plt.subplots(2, 1, figsize=(11, 4.2), sharex=True, gridspec_kw={"height_ratios": [1.4, 1.0]})
    t_ax = np.linspace(win_s, win_e, len(snippet))
    ax_wave.plot(t_ax, snippet, color="#1f77b4", linewidth=0.8)
    sub_runs = intervals_df[(intervals_df["end"] >= win_s) & (intervals_df["start"] <= win_e)]
    for _, r in sub_runs.iterrows():
        ax_wave.axvspan(max(win_s, r["start"]), min(win_e, r["end"]), color="#d4edda" if r["state"] == "SPEECH" else "#e2e3e5", alpha=0.45)
    ax_wave.set_ylabel("Amplitude"); ax_wave.set_title("Voice Activity Overlay • synchronized cursor", fontsize=10, fontweight="bold")
    ax_wave.grid(True, linestyle=":", alpha=0.4)
    k0, k1 = max(0, int(win_s / 0.010)), min(len(times_arr), int(win_e / 0.010))
    sub_e = energy_db_arr[k0:k1]
    if k1 > k0: ax_energy.plot(times_arr[k0:k1], sub_e, color="#2ca02c", linewidth=1.1, label="Short-Time Energy (dB)")
    ax_energy.axhline(thresh_db, color="#e0a800", linestyle=":", linewidth=1.6, label=f"Silence threshold ({thresh_db:.1f} dB)")
    ax_energy.set_ylabel("Energy (dB)"); ax_energy.set_xlabel("Time (seconds)"); ax_energy.set_xlim(win_s, win_e)
    ax_energy.legend(loc="lower right", fontsize=8); ax_energy.grid(True, linestyle=":", alpha=0.4)
    plt.tight_layout()
    pos_w, pos_e = ax_wave.get_position(), ax_energy.get_position()
    x0, x1 = float(pos_w.x0), float(pos_w.x1); y_top, y_bot = float(1.0 - pos_w.y1), float(1.0 - pos_e.y0)

    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110); plt.close(fig)
    b64_img = base64.b64encode(buf.getvalue()).decode("ascii")
    abuf = io.BytesIO(); sf.write(abuf, snippet, sr, format="WAV", subtype="PCM_16")
    b64_aud = base64.b64encode(abuf.getvalue()).decode("ascii")
    intvs_json = json.dumps([{"start": round(float(r["start"]), 3), "end": round(float(r["end"]), 3),
                              "duration": round(float(r["duration"]), 3), "state": str(r["state"])} for _, r in sub_runs.iterrows()])
    energy_json = json.dumps(np.round(np.clip(sub_e, -65.0, 0.0), 1).tolist())

    inner_html = f'''<!DOCTYPE html><html><head><meta charset="utf-8">
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: sans-serif; background: #fff; padding: 6px 10px; }}
.badge {{ padding: 7px 12px; border-radius: 4px; font-weight: bold; margin-bottom: 8px; font-size: 13px; }}
.badge-speech {{ background: #d4edda; color: #155724; border-left: 5px solid #28a745; }}
.badge-silence {{ background: #e2e3e5; color: #383d41; border-left: 5px solid #6c757d; }}
.plot-box {{ position: relative; width: 100%; cursor: crosshair; user-select: none; }}
.plot-img {{ width: 100%; display: block; }}
.cursor {{ position: absolute; top: {y_top*100:.2f}%; height: {(y_bot - y_top)*100:.2f}%; width: 0; border-left: 2px dashed #dc3545; pointer-events: none; z-index: 10; }}
.pill {{ position: absolute; top: -20px; left: 0; transform: translateX(-50%); background: #dc3545; color: #fff; font-size: 10px; font-weight: bold; padding: 1px 6px; border-radius: 3px; white-space: nowrap; }}
.bar {{ display: flex; align-items: center; gap: 8px; margin-bottom: 6px; background: #f8f9fa; border: 1px solid #e9ecef; padding: 6px 10px; border-radius: 4px; }}
button {{ padding: 4px 10px; border: none; border-radius: 3px; font-weight: bold; font-size: 11px; cursor: pointer; }}
.btn-s {{ background: #6c757d; color: #fff; }} .btn-r {{ background: #28a745; color: #fff; }}
</style></head><body>
<div id="badge" class="badge badge-speech"></div>
<div class="bar">
  <audio id="aud" src="data:audio/wav;base64,{b64_aud}" controls preload="auto" style="flex:1; height:30px;"></audio>
  <button class="btn-s" id="b_b1">⏪ -1s</button><button class="btn-s" id="b_f1">⏩ +1s</button><button class="btn-r" id="b_re">🔄 Replay</button>
</div>
<div class="plot-box" id="pbox"><img class="plot-img" src="data:image/png;base64,{b64_img}" />
  <div class="cursor" id="cur"><div class="pill" id="pill">📍 00:00.00</div></div></div>
<script>
const aud=document.getElementById('aud'),cur=document.getElementById('cur'),pill=document.getElementById('pill'),badge=document.getElementById('badge'),pbox=document.getElementById('pbox');
const x0={x0},x1={x1},ws={win_s},wd={actual_dur},th={thresh_db},intvs={intvs_json},eng={energy_json};
function fmt(s){{const m=Math.floor(s/60),sc=s%60;return (m<10?'0':'')+m+':'+(sc<10?'0':'')+sc.toFixed(2);}}
function upd(rT){{
  rT=Math.max(0,Math.min(wd,rT));const tA=ws+rT;
  cur.style.left=(x0+(rT/wd)*(x1-x0))*100+'%';pill.textContent='📍 '+fmt(tA);
  let cI=null;for(let it of intvs){{if(it.start<=tA&&it.end>=tA){{cI=it;break;}}}}
  const eIdx=Math.max(0,Math.min(eng.length-1,Math.round(rT/0.010))),eVal=eng.length>0?eng[eIdx]:-50;
  const isSp=cI?(cI.state==='SPEECH'):(eVal>th);
  if(isSp){{badge.className='badge badge-speech';badge.innerHTML='🟢 <b>'+fmt(tA)+' ('+tA.toFixed(2)+'s) → ACTIVE SPEECH</b> | '+eVal.toFixed(1)+' dB';}}
  else{{badge.className='badge badge-silence';badge.innerHTML='⚪ <b>'+fmt(tA)+' ('+tA.toFixed(2)+'s) → SILENCE / PAUSE</b> | '+eVal.toFixed(1)+' dB (safe cut zone)';}}
}}
let aId=null;function loop(){{if(!aud.paused&&!aud.ended){{upd(aud.currentTime);aId=requestAnimationFrame(loop);}}}}
aud.addEventListener('play',()=>{{if(aId)cancelAnimationFrame(aId);aId=requestAnimationFrame(loop);}});
aud.addEventListener('pause',()=>{{if(aId)cancelAnimationFrame(aId);upd(aud.currentTime);}});
aud.addEventListener('timeupdate',()=>{{if(aud.paused)upd(aud.currentTime);}});
pbox.addEventListener('click',(e)=>{{const r=pbox.getBoundingClientRect(),rat=(e.clientX-r.left)/r.width;
  if(rat>=x0&&rat<=x1){{const tR=((rat-x0)/(x1-x0))*wd;aud.currentTime=tR;upd(tR);}}}});
document.getElementById('b_b1').onclick=()=>{{aud.currentTime=Math.max(0,aud.currentTime-1);upd(aud.currentTime);}};
document.getElementById('b_f1').onclick=()=>{{aud.currentTime=Math.min(wd,aud.currentTime+1);upd(aud.currentTime);}};
document.getElementById('b_re').onclick=()=>{{aud.currentTime=0;upd(0);aud.play();}};
upd(0);
</script></body></html>'''
    return f'<iframe srcdoc="{html.escape(inner_html)}" style="width:100%; height:490px; border:1px solid #ced4da; border-radius:6px;"></iframe>'


def launch_timeline_pointer_inspector(audio_samples, pauses_df=None, vad_timeline_df=None, sr=DEFAULT_SR):
    total_dur = len(audio_samples) / sr
    intervals_df, times, energy_db, thresh_db = compute_instantaneous_energy_vad_overlay(audio_samples, sr, vad_timeline_df)
    sl = widgets.FloatSlider(value=0.0, min=0.0, max=total_dur, step=1.0, description="Seek (s):", layout=widgets.Layout(width="70%"))
    out = widgets.Output()
    def refresh(ch=None):
        out.clear_output(wait=True)
        with out: display(HTML(build_synchronized_player_iframe(audio_samples, sr, sl.value, 10.0, intervals_df, times, energy_db, thresh_db, total_dur)))
    sl.observe(refresh, names="value"); display(widgets.VBox([sl, out])); refresh()


def launch_spectrogram_region_editor(audio_samples, regions_df, video_dir=None, sr=DEFAULT_SR, user="annotator"):
    """Spectrogram with region overlays + human override: relabel, MOVE BORDERS, split, merge, delete, add (paint).
    Every edit is saved to regions.csv and logged to changes.csv with the annotator name. Returns the state dict."""
    total = len(audio_samples) / sr
    state = {"df": _renorm(regions_df.copy())}
    user_w = widgets.Text(value=user, description="Annotator:", layout=widgets.Layout(width="260px"))
    win_w = widgets.IntSlider(value=60, min=10, max=300, step=10, description="View (s):")
    start_w = widgets.FloatSlider(value=0, min=0, max=max(1.0, total - 10), step=5.0, description="From (s):", layout=widgets.Layout(width="70%"))
    reg_w = widgets.Dropdown(description="Region:", layout=widgets.Layout(width="70%"))
    lbl_w = widgets.RadioButtons(options=CONTENT_LABELS, description="Label:")
    ns_w, ne_w = widgets.FloatText(description="Start:", step=0.05), widgets.FloatText(description="End:", step=0.05)
    sp_w = widgets.FloatText(description="Split at:", step=0.05)
    pa_w, pb_w = widgets.FloatText(description="Paint from:", step=0.05), widgets.FloatText(description="to:", step=0.05)
    pl_w = widgets.Dropdown(options=CONTENT_LABELS, description="as:")
    note_w = widgets.Text(placeholder="Reason...", description="Note:", layout=widgets.Layout(width="60%"))
    msg, plot_out, audio_out = widgets.Output(), widgets.Output(), widgets.Output()
    mk = lambda d, s="": widgets.Button(description=d, button_style=s, layout=widgets.Layout(width="auto"))
    b_lbl, b_mv, b_split, b_merge, b_del, b_paint = mk("Set label", "success"), mk("Apply borders", "warning"), mk("Split"), mk("Merge with next"), mk("Delete"), mk("Paint interval")
    b_play, b_pb1, b_pb2 = mk("▶ Region"), mk("▶ around start (±3 s)"), mk("▶ around end (±3 s)")
    busy = {"v": False}

    def opts():
        d = state["df"]
        return [(f"#{i} {r['start']:.2f}-{r['end']:.2f}s ({r['duration']:.1f}s) {r['label'].upper()} [{r.get('status','')}]", i) for i, r in d.iterrows()]

    def fill_fields(*_):
        if busy["v"] or reg_w.value is None: return
        r = state["df"].loc[reg_w.value]
        lbl_w.value = r["label"]; ns_w.value, ne_w.value = float(r["start"]), float(r["end"])
        sp_w.value = round((r["start"] + r["end"]) / 2, 2); pa_w.value, pb_w.value = float(r["start"]), float(r["end"])
        start_w.value = float(np.clip(r["start"] - 5, 0, max(1.0, total - 10)))
        redraw()

    def redraw(*_):
        plot_out.clear_output(wait=True)
        with plot_out:
            fig = plot_spectrogram_with_regions(audio_samples, state["df"], start_w.value, start_w.value + win_w.value, sr, reg_w.value)
            plt.show()

    def commit(new_df, field, old, new, idx):
        state["df"] = new_df
        if video_dir:
            new_df.to_csv(Path(video_dir) / "regions.csv", index=False)
            log_human_change(video_dir, user_w.value, f"region_{idx}", field, old, new, note_w.value)
        busy["v"] = True; reg_w.options = opts(); reg_w.value = min(idx, len(new_df) - 1); busy["v"] = False
        fill_fields()

    def guard(fn):
        def h(_):
            msg.clear_output()
            try: fn()
            except Exception as e:
                with msg: print(f"❌ {e}")
        return h

    def r_(): return reg_w.value, state["df"].loc[reg_w.value]
    def on_lbl():
        i, r = r_(); commit(region_set_label(state["df"], i, lbl_w.value), "label", r["label"], lbl_w.value, i)
    def on_mv():
        i, r = r_(); commit(region_move_borders(state["df"], i, ns_w.value, ne_w.value), "borders", f"{r['start']}-{r['end']}", f"{ns_w.value}-{ne_w.value}", i)
    def on_split():
        i, r = r_(); commit(region_split(state["df"], i, sp_w.value), "split", f"{r['start']}-{r['end']}", f"at {sp_w.value}", i)
    def on_merge():
        i, r = r_(); commit(region_merge_with_next(state["df"], i), "merge_next", r["end"], "merged", i)
    def on_del():
        i, r = r_(); commit(region_delete(state["df"], i), "delete", f"{r['label']} {r['start']}-{r['end']}", "absorbed", max(0, i - 1))
    def on_paint():
        i, r = r_(); commit(region_paint(state["df"], pa_w.value, pb_w.value, pl_w.value), "paint", "-", f"{pa_w.value}-{pb_w.value}={pl_w.value}", i)
    def play(a, b):
        audio_out.clear_output(wait=True)
        with audio_out: display(Audio(audio_samples[int(max(0, a) * sr):int(min(total, b) * sr)], rate=sr))
    b_lbl.on_click(guard(on_lbl)); b_mv.on_click(guard(on_mv)); b_split.on_click(guard(on_split))
    b_merge.on_click(guard(on_merge)); b_del.on_click(guard(on_del)); b_paint.on_click(guard(on_paint))
    b_play.on_click(lambda _: play(*[state["df"].loc[reg_w.value, k] for k in ("start", "end")]))
    b_pb1.on_click(lambda _: play(state["df"].loc[reg_w.value, "start"] - 3, state["df"].loc[reg_w.value, "start"] + 3))
    b_pb2.on_click(lambda _: play(state["df"].loc[reg_w.value, "end"] - 3, state["df"].loc[reg_w.value, "end"] + 3))
    reg_w.observe(fill_fields, names="value"); start_w.observe(redraw, names="value"); win_w.observe(redraw, names="value")

    reg_w.options = opts()
    ui = widgets.VBox([widgets.HBox([user_w, note_w]), widgets.HBox([start_w, win_w]), plot_out, reg_w,
                       widgets.HBox([lbl_w, widgets.VBox([widgets.HBox([ns_w, ne_w, b_mv]), widgets.HBox([sp_w, b_split, b_merge, b_del]),
                                                          widgets.HBox([pa_w, pb_w, pl_w, b_paint])])]),
                       widgets.HBox([b_lbl, b_play, b_pb1, b_pb2]), audio_out, msg])
    display(ui); fill_fields()
    state["w"] = dict(reg=reg_w, lbl=lbl_w, ns=ns_w, ne=ne_w, sp=sp_w, pa=pa_w, pb=pb_w, pl=pl_w, msg=msg, b_lbl=b_lbl,
                      b_mv=b_mv, b_split=b_split, b_merge=b_merge, b_del=b_del, b_paint=b_paint)
    return state


def launch_region_inspector(audio_samples, regions_df, video_dir=None, sr=DEFAULT_SR, user="annotator"):
    return launch_spectrogram_region_editor(audio_samples, regions_df, video_dir, sr, user)


def launch_vad_overlay_inspector(audio_samples, pauses_df=None, vad_timeline_df=None, sr=DEFAULT_SR):
    if pauses_df is None or pauses_df.empty:
        print("No pauses available to inspect."); return
    opts = [(f"{p['pause_id']} ({p['start']:.2f}-{p['end']:.2f}s | {p['duration_ms']:.0f} ms) → {'SAFE' if p['is_safe'] else 'SHORT'}", i) for i, p in pauses_df.iterrows()]
    sel = widgets.Dropdown(options=opts, description="Pause:", layout=widgets.Layout(width="60%")); out = widgets.Output()
    def upd(ch=None):
        out.clear_output(); p = pauses_df.loc[sel.value]
        with out:
            t0, t1 = max(0.0, p["start"] - 2.0), min(len(audio_samples) / sr, p["end"] + 2.0)
            print(f"Pause {p['pause_id']} ({p['duration_ms']:.0f} ms) — context {t0:.2f}-{t1:.2f}s")
            display(Audio(audio_samples[int(t0 * sr):int(t1 * sr)], rate=sr))
    sel.observe(upd, names="value"); display(widgets.VBox([sel, out])); upd()


def launch_segment_inspector(audio_samples, segments_df, video_dir=None, sr=DEFAULT_SR, user="annotator"):
    """Review queue: listen, relabel (speech/music/mixed), mark reviewed. Saves segments.csv and logs every change."""
    f_drop = widgets.Dropdown(options=["All Segments", "⚠️ Review queue (LOW certainty / fallback)", "Fallback cuts (15-18 s)", "Not margin-compliant"], description="Filter:", layout=widgets.Layout(width="45%"))
    s_drop = widgets.Dropdown(description="Segment:", layout=widgets.Layout(width="60%"))
    user_w = widgets.Text(value=user, description="Annotator:"); note_w = widgets.Text(description="Note:", layout=widgets.Layout(width="50%"))
    lbl_w = widgets.RadioButtons(options=["speech", "music", "mixed"], description="Label:")
    b_save = widgets.Button(description="💾 Save (mark reviewed)", button_style="success"); out = widgets.Output()

    def refresh_list(ch=None):
        d, f = segments_df, f_drop.value
        if f.startswith("⚠️"): d = d[d["needs_review"] == 1]
        elif f.startswith("Fallback"): d = d[d["fallback_used"] == 1]
        elif f.startswith("Not margin"): d = d[d["margin_ok"] == 0]
        o = [(f"Seg {r['segment_id']} ({r['start']:.1f}-{r['end']:.1f}s | {r['duration']:.1f}s) [{r['cut_tier']}] {r['label']}", i) for i, r in d.iterrows()]
        s_drop.options = o if o else [("No matching segments", -1)]
        if o: s_drop.value = o[0][1]

    def update_seg(ch=None):
        out.clear_output()
        if s_drop.value is None or s_drop.value < 0: return
        r = segments_df.loc[s_drop.value]
        lbl_w.value = r["label"] if r["label"] in ("speech", "music", "mixed") else "speech"
        with out:
            if r["needs_review"] == 1:
                display(HTML(f'<div style="background:#fee2e2;border-left:5px solid #dc2626;padding:8px 12px;border-radius:4px;font-weight:bold;color:#991b1b;margin-bottom:8px;">⚠️ HUMAN REVIEW — certainty {r["cut_certainty"]:.2f} | cut: {r["cut_type"]} | lead {r["lead_ms"]} ms | trail {r["trail_ms"]} ms | margins ok: {bool(r["margin_ok"])}</div>'))
            print(f"Segment {r['segment_id']}: {r['start']:.2f}-{r['end']:.2f}s ({r['duration']:.2f}s) | {r['label'].upper()} (conf {r['label_conf']:.2f}) | status: {r['status']}")
            display(Audio(audio_samples[int(r["start"] * sr):int(r["end"] * sr)], rate=sr))

    def on_save(_):
        i = s_drop.value
        if i is None or i < 0: return
        old = segments_df.loc[i, "label"]
        if old != lbl_w.value:
            segments_df.loc[i, "label"] = lbl_w.value; segments_df.loc[i, "label_conf"] = 1.0
            if video_dir: log_human_change(video_dir, user_w.value, segments_df.loc[i, "segment_id"], "label", old, lbl_w.value, note_w.value)
        old_n = segments_df.loc[i, "needs_review"]
        segments_df.loc[i, "needs_review"] = 0; segments_df.loc[i, "status"] = "reviewed_student1"
        if video_dir:
            if old_n != 0: log_human_change(video_dir, user_w.value, segments_df.loc[i, "segment_id"], "needs_review", old_n, 0, note_w.value)
            segments_df.to_csv(Path(video_dir) / "segments.csv", index=False)
        with out: print("💾 saved")

    f_drop.observe(refresh_list, names="value"); s_drop.observe(update_seg, names="value"); b_save.on_click(on_save)
    display(widgets.VBox([widgets.HBox([f_drop, s_drop]), widgets.HBox([user_w, note_w]), widgets.HBox([lbl_w, b_save]), out]))
    refresh_list(); update_seg()

print("✅ Inspectors ready.")

In [ ]:
# 📊 Cell 8: Quality Control, Manual 10-minute Check, Summaries, Manifest & Safe Splitting

def validate_segments_integrity(df, video_dir=None, sr=DEFAULT_SR):
    """Real, countable checks. 'passed' = no failed check; warnings are informational."""
    checks = []
    def add(name, ok, detail): checks.append({"check": name, "passed": bool(ok), "detail": detail})
    if df.empty:
        return {"passed": False, "status": "FAILED", "checks": [], "errors": ["empty"], "warnings": [], "summary": {"passed_checks": 0, "total_checks": 0}}
    cut = df[df["cut_type"] != "music_whole"].sort_values("start")
    allv = df.sort_values("start")
    ov = [(a["segment_id"], b["segment_id"]) for a, b in zip(allv.to_dict("records")[:-1], allv.to_dict("records")[1:]) if a["end"] > b["start"] + 0.001]
    add("no overlapping segments", not ov, f"{len(ov)} overlaps {ov[:3]}")
    oob = cut[(cut.duration > FB_LEN + 0.02) | ((cut.duration < MIN_LEN - 0.02) & (cut.plan_tag == "ok"))]
    add("planned segments within 5-18 s", oob.empty, f"{len(oob)} out of bounds")
    pri = float(((cut.duration >= MIN_LEN - 0.01) & (cut.duration <= MAX_LEN + 0.01)).mean()) if len(cut) else 0
    add("priority window 5-15 s >= 90%", pri >= 0.90, f"{pri * 100:.1f}%")
    mg = float(cut["margin_ok"].mean()) if len(cut) else 0
    add("100-200 ms margins >= 80%", mg >= 0.80, f"{mg * 100:.1f}%")
    bad_lbl = df[~df["label"].isin(["speech", "music", "mixed"])]
    add("labels in {speech, music, mixed}", bad_lbl.empty, f"{len(bad_lbl)} invalid")
    miss, mism = [], []
    if video_dir is not None:
        for r in df.itertuples():
            p = Path(video_dir) / r.wav_file
            if not p.exists(): miss.append(r.segment_id); continue
            if abs(sf.info(str(p)).duration - r.duration) > 0.05: mism.append(r.segment_id)
    add("wav files exist and match durations", not miss and not mism, f"{len(miss)} missing, {len(mism)} duration mismatches")
    rev = float(cut["needs_review"].mean() * 100) if len(cut) else 0
    warnings_ = []
    if not (5.0 <= rev <= 15.0): warnings_.append(f"Review workload {rev:.1f}% is outside the 5-15% target.")
    if (cut.plan_tag != "ok").any(): warnings_.append(f"{int((cut.plan_tag != 'ok').sum())} segments from short_stream/relaxed/equal_split plans (flagged for review).")
    npass = sum(c["passed"] for c in checks)
    return {"passed": npass == len(checks), "status": "PASSED" if npass == len(checks) else "FAILED", "checks": checks,
            "errors": [f"{c['check']}: {c['detail']}" for c in checks if not c["passed"]], "warnings": warnings_,
            "total_segments": len(df), "total_duration_sec": round(float(df.duration.sum()), 2),
            "summary": {"passed_checks": npass, "total_checks": len(checks)}}


def generate_qc_markdown_report(segments_df, report_path="qc_report.md", video_dir=None, qc=None):
    """Only MEASURED numbers. The greedy-vs-DP table comes from ablation_metrics.json (both planners run on the same
    candidates in the same streams); it is omitted if that file does not exist."""
    report_path = Path(report_path); report_path.parent.mkdir(parents=True, exist_ok=True)
    cut = segments_df[segments_df["cut_type"] != "music_whole"]; n = max(1, len(cut))
    in_w = int(((cut.duration >= MIN_LEN - 0.01) & (cut.duration <= MAX_LEN + 0.01)).sum())
    L = ["# Audio Labelling QC Report (Student 1)",
         f"**Segments:** {len(segments_df)} ({len(cut)} speech/mixed cuts + {len(segments_df) - len(cut)} music regions) | **Total:** {segments_df.duration.sum() / 60:.2f} min\n",
         "## 1. Measured compliance (speech/mixed cuts)",
         f"- 5-15 s priority window: `{in_w}/{len(cut)} ({100 * in_w / n:.1f}%)`",
         f"- 15-18 s fallback: `{int(cut.fallback_used.sum())}/{len(cut)}`",
         f"- 100-200 ms margins satisfied: `{100 * cut.margin_ok.mean():.1f}%`" if len(cut) else "",
         f"- Cut tiers: HIGH `{int((cut.cut_tier == 'HIGH').sum())}`, MEDIUM `{int((cut.cut_tier == 'MEDIUM').sum())}`, LOW `{int((cut.cut_tier == 'LOW').sum())}`",
         f"- Human review workload: `{100 * cut.needs_review.mean():.1f}%` (target 5-15%)\n" if len(cut) else ""]
    if qc:
        L += ["## 2. Integrity checks", *[f"- {'✅' if c['passed'] else '❌'} {c['check']} — {c['detail']}" for c in qc["checks"]],
              *[f"- ⚠️ {w}" for w in qc["warnings"]], ""]
    ab = Path(video_dir or ".") / "ablation_metrics.json"
    if ab.exists():
        m = json.loads(ab.read_text(encoding="utf-8"))
        g, d = m["greedy"], m["dp"]
        if d["n"] and g["n"]:
            pc = lambda x, k: f"{100 * x[k] / max(1, x['n']):.1f}%"
            L += ["## 3. Ablation (measured): greedy baseline vs. global DP, same candidates, same streams",
                  "| Metric | Greedy | Global DP |", "| :-- | --: | --: |",
                  f"| Segments | {g['n']} | {d['n']} |", f"| In 5-15 s | {pc(g, 'in_5_15')} | {pc(d, 'in_5_15')} |",
                  f"| < 5 s | {g['under_5']} | {d['under_5']} |", f"| > 15 s | {g['over_15']} | {d['over_15']} |",
                  f"| Margin 100-200 ms OK | {pc(g, 'margin_ok')} | {pc(d, 'margin_ok')} |",
                  f"| Blind cuts (no pause) | {g['blind_cuts']} | {d['blind_cuts']} |", f"| Weak (valley) cuts | {g['weak_cuts']} | {d['weak_cuts']} |", ""]
    L.append("_Word-clip rate and label accuracy are NOT claimed here: measure them with the 10-minute manual check._")
    md = "\n".join(L); report_path.write_text(md, encoding="utf-8"); return md


def sample_manual_check_window(segments_df, video_dir, minutes=10.0, seed=0):
    """Pick a random contiguous ~10-minute stretch of segments for the manual check required by the brief."""
    rng = np.random.default_rng(seed); d = segments_df.sort_values("start").reset_index(drop=True)
    tot = d.duration.sum()
    for _ in range(50):
        i = int(rng.integers(0, len(d))); cum = d.duration.iloc[i:].cumsum()
        if cum.iloc[-1] >= minutes * 60 or tot < minutes * 60: break
    j = i + int((cum <= minutes * 60).sum()) + 1
    chk = d.iloc[i:j][["segment_id", "start", "end", "duration", "label", "cut_type", "cut_tier", "cut_certainty"]].copy()
    chk["label_wrong"], chk["cut_inside_word"], chk["checked"] = 0, 0, 0
    chk.to_csv(Path(video_dir) / "manual_check.csv", index=False)
    print(f"Manual-check sample: {len(chk)} segments, {chk.duration.sum() / 60:.1f} min -> manual_check.csv")
    return chk


def launch_manual_check(audio_samples, video_dir, sr=DEFAULT_SR):
    path = Path(video_dir) / "manual_check.csv"; chk = pd.read_csv(path)
    sel = widgets.Dropdown(options=[(f"{r.segment_id} [{r.cut_tier}] {r.start:.1f}-{r.end:.1f}s {'✔' if r.checked else ''}", i) for i, r in chk.iterrows()], description="Seg:", layout=widgets.Layout(width="60%"))
    c_lbl, c_cut = widgets.Checkbox(description="Region label is WRONG"), widgets.Checkbox(description="Cut falls INSIDE a word")
    b = widgets.Button(description="💾 Save & next", button_style="success"); out = widgets.Output()
    def show(_=None):
        out.clear_output(); r = chk.loc[sel.value]; c_lbl.value, c_cut.value = bool(r.label_wrong), bool(r.cut_inside_word)
        with out: display(Audio(audio_samples[int(r.start * sr):int(r.end * sr)], rate=sr))
    def save(_):
        i = sel.value; chk.loc[i, ["label_wrong", "cut_inside_word", "checked"]] = [int(c_lbl.value), int(c_cut.value), 1]
        chk.to_csv(path, index=False)
        sel.options = [(f"{r.segment_id} [{r.cut_tier}] {r.start:.1f}-{r.end:.1f}s {'✔' if r.checked else ''}", k) for k, r in chk.iterrows()]
        if i + 1 < len(chk): sel.value = i + 1
    sel.observe(show, names="value"); b.on_click(save); display(widgets.VBox([sel, widgets.HBox([c_lbl, c_cut, b]), out])); show()


def summarize_manual_check(video_dir):
    """Counts for the report + calibration of the certainty tiers (how often HIGH/MEDIUM/LOW cuts were really bad)."""
    chk = pd.read_csv(Path(video_dir) / "manual_check.csv"); done = chk[chk.checked == 1]
    print(f"Checked {len(done)}/{len(chk)} segments ({done.duration.sum() / 60:.1f} min)")
    print(f"• wrong region labels : {int(done.label_wrong.sum())} ({100 * done.label_wrong.mean():.1f}%)" if len(done) else "")
    print(f"• cuts inside a word  : {int(done.cut_inside_word.sum())} ({100 * done.cut_inside_word.mean():.1f}%)" if len(done) else "")
    if len(done):
        print("\nCut-tier calibration (share of cuts inside a word):")
        display(done.groupby("cut_tier")["cut_inside_word"].agg(["count", "sum", "mean"]).rename(columns={"sum": "bad", "mean": "bad_rate"}))
    return done


def video_summary(regions_df, segments_df, info=None, video_dir=None):
    reg = regions_df.assign(d=regions_df["end"] - regions_df["start"]).groupby("label")["d"].sum() / 60.0
    cut = segments_df[segments_df["cut_type"] != "music_whole"]
    row = {"video": (info or {}).get("title") or (info or {}).get("video_id"), "duration_min": round(regions_df["end"].max() / 60, 2),
           "speech_min": round(reg.get("speech", 0), 2), "music_min": round(reg.get("music", 0), 2), "mixed_min": round(reg.get("mixed", 0), 2),
           "silence_min": round(reg.get("silence", 0), 2), "segments": len(segments_df), "cut_segments": len(cut),
           "mean_seg_s": round(cut.duration.mean(), 2) if len(cut) else 0, "review_pct": round(100 * cut.needs_review.mean(), 1) if len(cut) else 0,
           "license": (info or {}).get("license"), "url": (info or {}).get("url")}
    df = pd.DataFrame([row])
    if video_dir: df.to_csv(Path(video_dir) / "video_summary.csv", index=False, encoding="utf-8")
    return df


def build_corpus_manifest(data_root, out_csv=None):
    """Concatenate every video's segments.csv (for Students 2 and 3 and the final report)."""
    parts = [pd.read_csv(p) for p in sorted(Path(data_root).glob("*/segments.csv"))]
    if not parts: raise FileNotFoundError("No segments.csv found")
    m = pd.concat(parts, ignore_index=True)
    if out_csv: m.to_csv(out_csv, index=False)
    return m


def split_corpus(manifest_df, out_dir="splits", ratios=(0.70, 0.15, 0.15)):
    """Optional (the brief does not require splits). >= 3 videos: whole videos go to one split (no leakage).
    < 3 videos: contiguous time blocks inside each video; NOT speaker/content independent, so say so in the report."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    vids = list(manifest_df["video_id"].unique())
    if len(vids) >= 3:
        order = {v: ("train" if i % 3 == 0 else "val" if i % 3 == 1 else "test") for i, v in enumerate(vids)}
        sp = manifest_df.assign(split=manifest_df["video_id"].map(order))
    else:
        print("⚠️ Fewer than 3 videos: using time-block splits (possible speaker overlap between splits).")
        parts = []
        for v, g in manifest_df.groupby("video_id"):
            g = g.sort_values("start"); n = len(g); a, b = int(n * ratios[0]), int(n * (ratios[0] + ratios[1]))
            parts.append(g.assign(split=["train"] * a + ["val"] * (b - a) + ["test"] * (n - b)))
        sp = pd.concat(parts)
    res = {k: sp[sp.split == k].drop(columns="split") for k in ("train", "val", "test")}
    for k, d in res.items(): d.to_csv(out_dir / f"{k}.csv", index=False, encoding="utf-8")
    return res

print("✅ QC / summary module ready.")


---
# 🚀 Execution Phase
Run the cells below in order. Edit the configuration cell first.

In [ ]:
# ⚙️ Configuration
USE_MOCK_DEMO = False          # True = synthetic audio, offline smoke test (also enables the heuristic AST fallback)
YOUTUBE_URL = "https://www.youtube.com/watch?v=w-aamvlT5C4"
LOCAL_FILE = None              # e.g. "/content/my_lecture.mp4" to skip yt-dlp (use this if YouTube blocks Colab)
COOKIES_FILE = None            # path to a cookies.txt exported from your browser (helps with Colab bot-blocking)
USE_DRIVE = False              # True: keep data on Google Drive so a Colab disconnect does not lose an hour of work
ANNOTATOR = "your_name"        # written to changes.csv for every human edit
USE_WORD_VERIFICATION = True   # faster-whisper word-boundary check (needs GPU for speed)
WORD_MODEL_SIZE = "small"      # "tiny"/"base" on CPU; "small"/"medium" on GPU

ALLOW_HEURISTIC_FALLBACK = USE_MOCK_DEMO
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/speech_corpus/data")
else:
    DATA_ROOT = Path("data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 📥 Steps 1 & 2: Download + audio extraction (+ info.json)
if USE_MOCK_DEMO:
    video_id = "mock_demo_audio"; video_dir = DATA_ROOT / video_id; video_dir.mkdir(exist_ok=True)
    audio_samples, gt_df = build_mock_audio(42, DEFAULT_SR)
    write_wav(video_dir / "audio.wav", audio_samples, DEFAULT_SR)
    info = _save_info(video_dir, {"video_id": video_id, "source": "mock", "url": None, "title": "mock", "license": "n/a", "duration_s": len(audio_samples) / DEFAULT_SR})
elif LOCAL_FILE:
    video_id = extract_video_id(str(LOCAL_FILE)); video_dir = DATA_ROOT / video_id
    audio_path, info = ingest_local_file(LOCAL_FILE, video_dir)
    audio_samples, _ = read_wav(audio_path, DEFAULT_SR)
else:
    video_id = extract_video_id(YOUTUBE_URL); video_dir = DATA_ROOT / video_id
    audio_path, info = download_video(YOUTUBE_URL, video_dir, cookies_file=COOKIES_FILE)
    audio_samples, _ = read_wav(audio_path, DEFAULT_SR)

dur_s = len(audio_samples) / DEFAULT_SR
print(f"{info.get('title')} | channel: {info.get('channel')} | license: {info.get('license')}")
print(f"Audio: {dur_s / 60:.2f} min @ {DEFAULT_SR} Hz | workspace: {video_dir}")
display(Audio(audio_samples[:int(min(30.0, dur_s) * DEFAULT_SR)], rate=DEFAULT_SR))

In [ ]:
# ⚡ Step 3.0: VAD pauses + adaptive silence floor (needed to snap region borders)
SILENCE_THRESHOLD_S = 0.25
vad_timeline_df, pauses_df = detect_speech_activity_overlay(audio_samples, DEFAULT_SR, SILENCE_THRESHOLD_S, min_pause_ms=120)
intervals_df, times_arr, energy_db_arr, thresh_db = compute_instantaneous_energy_vad_overlay(audio_samples, DEFAULT_SR, vad_timeline_df)
floor_db, _ = adaptive_silence_floor(energy_db_arr)
sp_time = vad_timeline_df.loc[vad_timeline_df.state == "SPEECH", "duration"].sum()
print(f"VAD backend: {VAD_BACKEND_USED['name']} | pauses: {len(pauses_df)} (>=250 ms: {int(pauses_df.is_safe.sum())})")
print(f"Speech activity {sp_time / 60:.1f} min ({100 * sp_time / dur_s:.1f}%) | silence floor {floor_db:.1f} dB, threshold {thresh_db:.1f} dB")
vad_timeline_df.to_csv(video_dir / "vad_timeline.csv", index=False)
pauses_df.to_csv(video_dir / "vad_pauses.csv", index=False)
intervals_df.to_csv(video_dir / "instant_overlay.csv", index=False)
display(pauses_df.head(8))

In [ ]:
# 🧠 Step 3: AST classification -> non-overlapping regions -> snapped to pauses
regions_csv, windows_csv = video_dir / "regions.csv", video_dir / "windows_ast.csv"
FORCE_RECLASSIFY = False
if not FORCE_RECLASSIFY and regions_csv.exists() and windows_csv.exists():
    print("⚡ Loading cached regions (delete regions.csv or set FORCE_RECLASSIFY=True to redo)")
    regions_df, windows_df = pd.read_csv(regions_csv), pd.read_csv(windows_csv)
else:
    regions_df, windows_df = classify_audio(audio_samples, DEFAULT_SR, pauses_df=pauses_df)
    regions_df.to_csv(regions_csv, index=False); windows_df.to_csv(windows_csv, index=False)
assert (regions_df["start"].values[1:] >= regions_df["end"].values[:-1] - 1e-6).all(), "regions overlap!"
print(f"{len(regions_df)} regions:"); display(regions_df)
print((regions_df.assign(m=regions_df.duration / 60).groupby("label")["m"].sum().round(2)).to_string(), "(minutes)")

In [ ]:
# 🔍 Inspector 1: Spectrogram + region editor (move borders / relabel / split / merge / delete / add)
region_editor = launch_spectrogram_region_editor(audio_samples, regions_df, video_dir, DEFAULT_SR, user=ANNOTATOR)
# Every edit is saved to regions.csv and logged in changes.csv; Step 4 reads regions.csv.

In [ ]:
# 📍 Inspector 2: Waveform + energy with a cursor synchronised to playback
launch_timeline_pointer_inspector(audio_samples, pauses_df, vad_timeline_df, DEFAULT_SR)

In [ ]:
# 🔍 Inspector 3: Pauses and safe zones
launch_vad_overlay_inspector(audio_samples, pauses_df, vad_timeline_df, DEFAULT_SR)

In [ ]:
# ✂️ Step 4: Cut Engine v2
segments_df = cut_speech_and_mixed_segments(audio_samples, regions_df, video_dir, DEFAULT_SR, pauses=pauses_df,
                                            use_word_verification=USE_WORD_VERIFICATION, word_model_size=WORD_MODEL_SIZE)
display(segments_df.head(10))

In [ ]:
# 🔍 Inspector 4: Review queue (relabel, mark reviewed; saved + logged)
launch_segment_inspector(audio_samples, segments_df, video_dir, DEFAULT_SR, user=ANNOTATOR)

In [ ]:
# 📋 Step 5: QC report (measured numbers only)
qc = validate_segments_integrity(segments_df, video_dir=video_dir)
md = generate_qc_markdown_report(segments_df, video_dir / "qc_report.md", video_dir=video_dir, qc=qc)
print(f"QC: {qc['status']} ({qc['summary']['passed_checks']}/{qc['summary']['total_checks']} checks)")
print(md)

In [ ]:
# 🧪 Cell 12: Ground-Truth Manual Quality Check UI (Full Video & Review Queue)
import ipywidgets as widgets
from IPython.display import display, Audio, HTML
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# 1. Prepare Ground-Truth Full Video Data (manual_check.csv)
# ---------------------------------------------------------
video_dir = Path(video_dir)
video_dir.mkdir(parents=True, exist_ok=True)
manual_csv_path = video_dir / "manual_check.csv"

# Load or initialize all segments from segments_df (Full Video, not just 10 mins)
if "segments_df" in globals() and segments_df is not None and len(segments_df) > 0:
    base_df = segments_df.copy()
else:
    raise ValueError("segments_df is empty or not defined! Please run Cell 9 first.")

# Ensure label column compatibility
if "label" not in base_df.columns and "primary_label" in base_df.columns:
    base_df["label"] = base_df["primary_label"]
elif "primary_label" not in base_df.columns and "label" in base_df.columns:
    base_df["primary_label"] = base_df["label"]

# Determine if a segment is flagged for Human Review (Labelling + Segmentation)
is_flagged_series = (
    (base_df.get("needs_review", 0) == 1) |
    (base_df.get("cut_tier", "HIGH").astype(str).str.upper() == "LOW") |
    (base_df.get("review_priority", "LOW").astype(str).str.upper().isin(["HIGH", "URGENT"])) |
    (base_df.get("confidence", 1.0).astype(float) < 0.60) |
    (base_df.get("cut_type", "").astype(str).str.lower() == "blind")
)
base_df["is_flagged"] = is_flagged_series

# If existing manual_check.csv exists, merge existing annotations
if manual_csv_path.exists():
    try:
        existing_chk = pd.read_csv(manual_csv_path)
        ann_cols = ["segment_id", "checked", "label_wrong", "corrected_label", "cut_inside_word", "cut_bad_boundary", "notes"]
        existing_ann = existing_chk[[c for c in ann_cols if c in existing_chk.columns]]
        if "segment_id" in existing_ann.columns and len(existing_ann) > 0:
            base_df = base_df.merge(existing_ann, on="segment_id", how="left", suffixes=("", "_old"))
    except Exception as e:
        print(f"Notice: loading existing annotations: {e}")

# Ensure annotation columns exist
for col in ["checked", "label_wrong", "cut_inside_word", "cut_bad_boundary"]:
    if col not in base_df.columns:
        base_df[col] = 0
    base_df[col] = base_df[col].fillna(0).astype(int)

for col in ["corrected_label", "notes"]:
    if col not in base_df.columns:
        base_df[col] = ""
    base_df[col] = base_df[col].fillna("").astype(str)

# Save initialized full video check file
base_df.to_csv(manual_csv_path, index=False)
chk_df = pd.read_csv(manual_csv_path)

print(f"🎬 Loaded Full Video: {len(chk_df)} segments ({chk_df['duration'].sum()/60:.1f} min)")
print(f"🔍 Flagged for Human Review (Labelling & Segmentation): {int(chk_df['is_flagged'].sum())}/{len(chk_df)} segments")

# ---------------------------------------------------------
# 2. Build Interactive UI
# ---------------------------------------------------------
LABEL_COLORS = {
    "speech": "#2ca02c",
    "music": "#1f77b4",
    "mixed": "#ff7f0e",
    "silence": "#7f7f7f"
}

# 1. Scope selection: Flagged Only vs Full Video
n_flagged = int(chk_df["is_flagged"].sum())
scope_radio = widgets.RadioButtons(
    options=[
        ("🔍 Flagged for Human Review Only (Labelling & Segmentation)", "flagged"),
        ("🎬 All Segments (Full Video)", "all")
    ],
    value="flagged" if n_flagged > 0 else "all",
    description="Scope:",
    layout=widgets.Layout(width="100%")
)

stats_label = widgets.HTML()
sel = widgets.Dropdown(description="Segment:", layout=widgets.Layout(width="80%"))

# 2. Human Review Controls (Labelling & Segmentation)
c_lbl = widgets.Checkbox(description="❌ Region label is WRONG", value=False, indent=False, layout=widgets.Layout(width="220px"))
corrected_drop = widgets.Dropdown(
    options=["speech", "music", "mixed", "silence"],
    description="Correct To:",
    layout=widgets.Layout(width="240px")
)
c_cut_word = widgets.Checkbox(description="✂️ Cut falls INSIDE a word (Word split risk)", value=False, indent=False, layout=widgets.Layout(width="310px"))
c_cut_bad = widgets.Checkbox(description="⚠️ Unnatural / Bad Cut Boundary", value=False, indent=False, layout=widgets.Layout(width="280px"))
txt_notes = widgets.Text(placeholder="Optional notes or reason for error...", description="Notes:", layout=widgets.Layout(width="90%"))

b_save_next = widgets.Button(description="💾 Save & Next", button_style="success", icon="save", layout=widgets.Layout(width="150px"))
b_mark_good = widgets.Button(description="✔ Mark Correct & Next", button_style="primary", icon="check", layout=widgets.Layout(width="180px"))
b_prev = widgets.Button(description="⏮ Prev", layout=widgets.Layout(width="80px"))
b_next = widgets.Button(description="Next ⏭", layout=widgets.Layout(width="80px"))

info_out = widgets.Output()
audio_out = widgets.Output()

state = {
    "df": chk_df,
    "filtered_indices": []
}

def _sync_filter():
    df = state["df"]
    if scope_radio.value == "flagged":
        sub_idx = df[df["is_flagged"] == True].index.tolist()
        if not sub_idx:
            sub_idx = df.index.tolist()
    else:
        sub_idx = df.index.tolist()
    state["filtered_indices"] = sub_idx

    opts = []
    for i in sub_idx:
        r = df.loc[i]
        sid = r.get("segment_id", i + 1)
        lbl = str(r.get("label", r.get("primary_label", "speech"))).lower()
        tier = str(r.get("cut_tier", "HIGH")).upper()
        st, en, dur = float(r["start"]), float(r["end"]), float(r["duration"])
        is_fl = bool(r.get("is_flagged", False))
        is_done = bool(r.get("checked", 0) == 1)
        
        fl_tag = "⚠️" if is_fl else ""
        done_tag = "✔" if is_done else ""
        text = f"Seg {int(sid):04d} [{lbl.upper()}] [{tier}] {st:.1f}s–{en:.1f}s ({dur:.1f}s) {fl_tag} {done_tag}".strip()
        opts.append((text, i))
        
    sel.options = opts
    n_tot = len(df)
    n_fl = int(df["is_flagged"].sum())
    n_done = int((df["checked"] == 1).sum())
    stats_label.value = (
        f"<div style='font-size:13px; margin:4px 0; color:#555;'>"
        f"<b>Scope Segments:</b> {len(sub_idx)} &nbsp;|&nbsp; "
        f"<b>Flagged for Review (Labelling + Segmentation):</b> {n_fl}/{n_tot} ({100*n_fl/max(1,n_tot):.1f}%) &nbsp;|&nbsp; "
        f"<b>Checked:</b> {n_done}/{n_tot} ({100*n_done/max(1,n_tot):.1f}%)"
        f"</div>"
    )
    if sub_idx:
        if sel.value not in sub_idx:
            sel.value = sub_idx[0]

def show(_=None):
    info_out.clear_output()
    audio_out.clear_output()
    if sel.value is None or sel.value not in state["df"].index:
        return
        
    i = sel.value
    r = state["df"].loc[i]
    
    c_lbl.value = bool(r.get("label_wrong", 0))
    c_cut_word.value = bool(r.get("cut_inside_word", 0))
    c_cut_bad.value = bool(r.get("cut_bad_boundary", 0))
    txt_notes.value = str(r.get("notes", "")) if pd.notnull(r.get("notes")) else ""
    
    lbl = str(r.get("label", r.get("primary_label", "speech"))).lower()
    if pd.notnull(r.get("corrected_label")) and str(r.get("corrected_label")).strip():
        corrected_drop.value = str(r.get("corrected_label")).strip()
    else:
        corrected_drop.value = lbl if lbl in ["speech", "music", "mixed", "silence"] else "speech"

    color = LABEL_COLORS.get(lbl, "#333")
    tier = str(r.get("cut_tier", "HIGH")).upper()
    conf = float(r.get("confidence", 1.0)) if pd.notnull(r.get("confidence")) else 1.0
    cert = float(r.get("cut_certainty", 1.0)) if pd.notnull(r.get("cut_certainty")) else 1.0
    is_fl = bool(r.get("is_flagged", False))
    prio = str(r.get("review_priority", "LOW")).upper()
    rev_reason = str(r.get("review_reason", "none"))
    cut_type = str(r.get("cut_type", "midpoint"))
    v_agree = str(r.get("voting_agreement", "3/3"))
    lead_ms = int(r.get("lead_ms", 150)) if pd.notnull(r.get("lead_ms")) else 150
    trail_ms = int(r.get("trail_ms", 150)) if pd.notnull(r.get("trail_ms")) else 150
    
    status_badge = f"<span style='background:{'#d9534f' if is_fl else '#5cb85c'}; color:white; padding:2px 8px; border-radius:4px; font-weight:bold;'>{'⚠️ REVIEW REQUIRED' if is_fl else '✅ AUTO-ACCEPTED'}</span>"
    checked_badge = "<span style='background:#0275d8; color:white; padding:2px 8px; border-radius:4px;'>✔ CHECKED</span>" if r.get("checked", 0) == 1 else ""

    html_info = f'''
    <div style="border:1px solid #ccc; border-radius:8px; padding:12px; margin:8px 0; background-color:#fcfcfc;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
            <span style="font-size:16px; font-weight:bold;">Segment #{int(r.get('segment_id', i+1)):04d} &nbsp; ({float(r['start']):.2f}s – {float(r['end']):.2f}s, Dur: {float(r['duration']):.2f}s)</span>
            <div>{status_badge} &nbsp; {checked_badge}</div>
        </div>
        
        <div style="display:grid; grid-template-columns: 1fr 1fr; gap:10px; font-size:13px; line-height:1.6;">
            <div style="background:#fff; border:1px solid #e0e0e0; border-radius:6px; padding:8px 12px;">
                <div style="font-weight:bold; color:#333; margin-bottom:4px; border-bottom:1px solid #eee; padding-bottom:3px;">🏷️ SYSTEM LABELLING</div>
                <div><b>Predicted Label:</b> <span style="background:{color}; color:white; padding:2px 7px; border-radius:3px; font-weight:bold;">{lbl.upper()}</span></div>
                <div><b>Label Confidence:</b> {conf:.1%} &nbsp;|&nbsp; <b>Composition:</b> {r.get('label_composition', lbl)}</div>
                <div><b>Source Regions:</b> {r.get('source_region_ids', 'N/A')}</div>
            </div>
            
            <div style="background:#fff; border:1px solid #e0e0e0; border-radius:6px; padding:8px 12px;">
                <div style="font-weight:bold; color:#333; margin-bottom:4px; border-bottom:1px solid #eee; padding-bottom:3px;">✂️ SYSTEM SEGMENTATION</div>
                <div><b>Certainty Tier:</b> <span style="font-weight:bold;">{tier}</span> (Score: {cert:.2f}) &nbsp;|&nbsp; <b>Priority:</b> {prio}</div>
                <div><b>Cut Type:</b> {cut_type} &nbsp;|&nbsp; <b>Voting Agreement:</b> {v_agree}</div>
                <div><b>Silence Margins:</b> Lead: {lead_ms}ms, Trail: {trail_ms}ms &nbsp;|&nbsp; <b>Reason:</b> <code style="color:#c7254e;">{rev_reason}</code></div>
            </div>
        </div>
    </div>
    '''
    with info_out:
        display(HTML(html_info))
    with audio_out:
        sr_rate = DEFAULT_SR if "DEFAULT_SR" in globals() else 16000
        st_samp = int(float(r["start"]) * sr_rate)
        en_samp = int(float(r["end"]) * sr_rate)
        if "audio_samples" in globals() and audio_samples is not None:
            display(Audio(audio_samples[st_samp:en_samp], rate=sr_rate))

def _save_feedback(mark_correct=False):
    i = sel.value
    if i is None or i not in state["df"].index:
        return
        
    if mark_correct:
        lbl_w = 0
        cut_w = 0
        cut_b = 0
        corr_lbl = ""
    else:
        lbl_w = int(c_lbl.value)
        cut_w = int(c_cut_word.value)
        cut_b = int(c_cut_bad.value)
        corr_lbl = corrected_drop.value if lbl_w else ""
        
    notes_val = txt_notes.value.strip()
    state["df"].loc[i, ["checked", "label_wrong", "cut_inside_word", "cut_bad_boundary", "corrected_label", "notes"]] = [
        1, lbl_w, cut_w, cut_b, corr_lbl, notes_val
    ]
    state["df"].to_csv(manual_csv_path, index=False)
    
    cur_list = state["filtered_indices"]
    if i in cur_list:
        pos = cur_list.index(i)
        if pos + 1 < len(cur_list):
            sel.value = cur_list[pos + 1]
        else:
            _sync_filter()
    else:
        _sync_filter()

def on_prev(_):
    cur_list = state["filtered_indices"]
    if sel.value in cur_list:
        pos = cur_list.index(sel.value)
        if pos > 0:
            sel.value = cur_list[pos - 1]

def on_next(_):
    cur_list = state["filtered_indices"]
    if sel.value in cur_list:
        pos = cur_list.index(sel.value)
        if pos + 1 < len(cur_list):
            sel.value = cur_list[pos + 1]

scope_radio.observe(lambda _: _sync_filter(), names="value")
sel.observe(show, names="value")
b_save_next.on_click(lambda _: _save_feedback(mark_correct=False))
b_mark_good.on_click(lambda _: _save_feedback(mark_correct=True))
b_prev.on_click(on_prev)
b_next.on_click(on_next)

ui = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px;'>🧪 Ground-Truth Manual Quality Check (Full Video & Review Queue)</h3>"),
    scope_radio,
    stats_label,
    widgets.HBox([b_prev, sel, b_next]),
    info_out,
    audio_out,
    widgets.HTML("<div style='font-weight:bold; margin-top:8px; margin-bottom:4px;'>Human Reviewer Annotations:</div>"),
    widgets.HBox([c_lbl, corrected_drop]),
    widgets.HBox([c_cut_word, c_cut_bad]),
    txt_notes,
    widgets.HBox([b_save_next, b_mark_good], layout=widgets.Layout(margin="8px 0"))
])

_sync_filter()
show()
display(ui)


In [ ]:
summarize_manual_check(video_dir)     # run after you finished ticking the segments

In [ ]:
# 📂 Per-video summary + corpus manifest (+ optional split)
summary_df = video_summary(pd.read_csv(video_dir / "regions.csv"), segments_df, info, video_dir)
display(summary_df)
manifest = build_corpus_manifest(DATA_ROOT, DATA_ROOT / "corpus_manifest.csv")
print(f"Manifest: {len(manifest)} segments from {manifest.video_id.nunique()} video(s) -> {DATA_ROOT / 'corpus_manifest.csv'}")
DO_SPLIT = False     # the brief does not require train/val/test; enable only if you need it
if DO_SPLIT:
    splits = split_corpus(manifest, DATA_ROOT / "splits")

In [ ]:
# 💾 Step 7: Zip for hand-off to Student 2 / Student 3
import zipfile
zip_filename = f"{video_id}_corpus.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as z:
    for f in video_dir.rglob("*"):
        if f.is_file(): z.write(f, arcname=f.relative_to(video_dir.parent))
print(f"✅ {zip_filename} ({Path(zip_filename).stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zip_filename)